# Naming a City

### Live demo: https://lara-yeyati-preiss.github.io/naming-a-city/map.html

### Project Overview:

This project uses OpenStreetMap data and a hybrid text-analysis approach to explore what the names of Buenos Aires cafés, bars, libraries, and other key local venues might reveal about the city's identity.


### Some Research Questions:

- Which cultural traditions are most visible in Buenos Aires' commercial landscape?
- How do naming patterns correlate with neighborhood socioeconomic profiles?
- What does the linguistic diversity of business names reveal about the city's identity?


### Methodology Overview:

**Data Collection**  
- Query OpenStreetMap for culturally significant venue types (cafés, bars, restaurants, bookstores, cinemas, theaters)
- Filter to Ciudad Autónoma de Buenos Aires administrative boundary
- Extract venue names, coordinates, and metadata


**Spatial Enrichment**  
- Perform point-in-polygon spatial joins with neighborhood boundaries
- Attach comuna (administrative district) identifiers to each venue
- Merge socioeconomic data at the comuna level


**Cultural Motif Classification**  
- **Regex-based classification:** Apply regex to identify cultural motifs
- **LLM / manual review:** Apply LLM-based overrides for specific cases, followed with manual overrides and review
- **Chain detection:** Filter out chains (defined as >8 locations with identical names), to avoid them overpowering 
the analysis results


**Quantitative Analysis**  
- Compute motif prevalence by neighborhood and comuna
- Extract distinctive terms per motif using TF-IDF
- Correlate naming patterns with socioeconomic variables (income, age distribution)


**Visualization Preprocessing**
- Generate GeoJSON files for Mapbox
- Create JSON metadata files with motif descriptions, term frequencies, and correlations


## Step 1: Data Collection from OpenStreetMap

**Goal:** Extract venue data for a subset of culturally significant place types from OpenStreetMap using the Overpass API.

We will search for specific venue types that reflect everyday urban culture in Buenos Aires: cafés, bars, restaurants, bakeries, ice cream shops, bookstores, cinemas, and theaters. This approach will allow us to build a dataset that's small enough to conduct a detailed exploratory study and large enough to extract relevant insights from this exploration.

**Technical Approach:**
- Query Overpass API with administrative boundary filter (Ciudad Autónoma de Buenos Aires)
- Extract only venues with name tags
- Store raw results as CSV with coordinates, tags, and metadata

**Output:** `osm_naming_ba_porteno_raw.csv` — baseline dataset of named venues with geographic coordinates

In [31]:
# step 1: data collection from openstreetmap
# --------------------------------------------------------------------
# this cell queries the overpass API (https://wiki.openstreetmap.org/wiki/Overpass_API)
# to download venue data for culturally significant place types in CABA.

import os, time, math, requests, pandas as pd
from tqdm import tqdm

# define the porteño cultural universe

FILTERS_PORTEÑO = {
    "amenity": ["cafe", "bar", "pub", "nightclub", "restaurant", "cinema", "theatre"],
    "shop": ["ice_cream", "books", "bakery", "deli"],
}

# ensure output directory exists
os.makedirs("data/processed", exist_ok=True)

# output path for raw OSM data
OUT_CSV_RAW = "data/processed/osm_naming_ba_porteno_raw.csv"

# overpass API endpoints (using multiple mirrors reduces risk of downtime)
ENDPOINTS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

# configuration for "polite" API use
TIMEOUT_SEC = 90          # maximum wait time for one response
BACKOFF_BASE = 1.7        # base for exponential backoff between retries
SLEEP_AFTER_REQ = 0.5     # pause between requests to avoid hammering servers

def build_area_query(key: str, values: list[str]) -> str:
    """
    build an overpass query for one OSM key and list of values,
    returns all named nodes/ways/relations within CABA boundary
    """
    safe_vals = "|".join(sorted({v.replace("|", "\\|") for v in values}))
    return f"""
    [out:json][timeout:180];

    area
      ["name"="Ciudad Autónoma de Buenos Aires"]
      ["boundary"="administrative"];
    (._;>;);

    (
      node["{key}"~"^({safe_vals})$"]["name"](area);
      way["{key}"~"^({safe_vals})$"]["name"](area);
      relation["{key}"~"^({safe_vals})$"]["name"](area);
    );
    out tags center;
    """

def call_overpass(query: str):
    """
    POST to overpass API with exponential backoff,
    tries multiple endpoints before failing
    """
    last_err = None
    for url in ENDPOINTS:
        for attempt in range(1, 4):
            try:
                r = requests.post(url, data={"data": query}, timeout=TIMEOUT_SEC)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last_err = e
                time.sleep((BACKOFF_BASE ** attempt) * 0.6)
    raise RuntimeError(f"All overpass endpoints failed. Last error: {last_err}")

def chunks(seq, n):
    """split list into chunks of size n."""
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

# maximum values per query to avoid overpass timeouts
MAX_VALUES_PER_QUERY = 20

# main fetch loop
rows = []
total_chunks = sum(math.ceil(len(vs) / MAX_VALUES_PER_QUERY) for vs in FILTERS_PORTEÑO.values())
pbar = tqdm(total=total_chunks, desc="fetching our universe of venues", unit="q")

for key, values in FILTERS_PORTEÑO.items():
    for block in chunks(values, MAX_VALUES_PER_QUERY):
        q = build_area_query(key, block)
        try:
            data = call_overpass(q)
            for el in data.get("elements", []):
                tags = el.get("tags", {}) or {}
                name = (tags.get("name") or "").strip()
                if not name:
                    continue  # skip unnamed venues

                osm_type, osm_id = el.get("type"), el.get("id")

                # extract coordinates: nodes have lat/lon directly,
                # ways and relations store a "center" point
                if osm_type == "node":
                    lat, lon = el.get("lat"), el.get("lon")
                else:
                    c = el.get("center") or {}
                    lat, lon = c.get("lat"), c.get("lon")

                if lat is None or lon is None:
                    continue  # skip venues without coordinates

                matched_val = tags.get(key, "")

                # assemble flat record with core fields + helpful metadata
                rows.append({
                    "osm_type": osm_type,
                    "osm_id": osm_id,
                    "key": key,
                    "value": matched_val,
                    "name": name,
                    "lat": float(lat),
                    "lon": float(lon),
                    "addr:street": tags.get("addr:street", ""),
                    "addr:housenumber": tags.get("addr:housenumber", ""),
                    "brand": tags.get("brand", ""),
                    "operator": tags.get("operator", ""),
                    "website": tags.get("website", ""),
                    "cuisine": tags.get("cuisine", ""),
                    "opening_hours": tags.get("opening_hours", ""),
                    "description": tags.get("description", ""),
                })
        except Exception:
            pass  # silently skip failed queries
        finally:
            pbar.update(1)
            time.sleep(SLEEP_AFTER_REQ)

pbar.close()

# convert to dataframe and remove duplicate OSM elements
df_raw = pd.DataFrame(rows)
if not df_raw.empty:
    df_raw = df_raw.drop_duplicates(subset=["osm_type", "osm_id"]).reset_index(drop=True)

# save raw dataset
df_raw.to_csv(OUT_CSV_RAW, index=False)
print(f"porteño raw data saved → {OUT_CSV_RAW}  |  rows: {len(df_raw)}")


fetching our universe of venues: 100%|██████████| 2/2 [00:05<00:00,  2.98s/q]

porteño raw data saved → data/processed/osm_naming_ba_porteno_raw.csv  |  rows: 4922


## Step 2: Spatial Join — Attaching Administrative Geography

**Goal:** Enrich venue points with neighborhood (barrio) and district (comuna) identifiers through spatial joins.

To analyze naming patterns geographically, we need to associate each venue with its administrative context. Buenos Aires is divided into 15 comunas and 48 barrios. By attaching these identifiers to each venue, we can aggregate and compare naming patterns across administrative boundaries.

**Technical Approach:**
- Load venue points from Step 1
- Load barrio polygons from GCBA (Gobierno de la Ciudad de Buenos Aires) open data portal
- Perform point-in-polygon spatial join to attach neighborhood (barrio) name and comuna number to each venue
- Save as a CSV

**Data Source:** GCBA Barrios GeoJSON (https://cdn.buenosaires.gob.ar/datosabiertos/datasets/barrios/barrios.geojson)

**Output:** `osm_naming_ba_porteno_with_barrios.csv` — venues enriched with geographic attributes

In [32]:
# step 2: spatial join — attach barrio and comuna to each venue point
# --------------------------------------------------------------------

import geopandas as gpd

# input: raw OSM venues with coordinates
IN_CSV_POINTS = "data/processed/osm_naming_ba_porteno_raw.csv"

# output: venues enriched with neighborhood and comuna columns
OUT_CSV_BARRIOS = "data/processed/osm_naming_ba_porteno_with_barrios.csv"

# load points as regular dataframe
df = pd.read_csv(IN_CSV_POINTS)

# convert to geodataframe with point geometries from lon/lat
# using WGS84 (EPSG:4326) coordinate reference system
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

# load neighborhood (barrios) polygons downloaded from the city's open data portal to a local GeoJSON file
# source: https://cdn.buenosaires.gob.ar/datosabiertos/datasets/barrios/barrios.geojson
barrios = gpd.read_file("data/comunas_data/barrios.geojson")
# standardize column names to lowercase for easier matching
barrios.columns = [c.lower() for c in barrios.columns]

# keep only essential columns: barrio name, comuna code, geometry
barrios = barrios[["nombre", "comuna", "geometry"]].rename(
    columns={"nombre": "barrio"}
)

# spatial join: assign each point to its containing barrio polygon
# using 'within' predicate (point must be inside polygon)
gdf_join = gpd.sjoin(gdf, barrios, how="left", predicate="within").drop(columns=["index_right"])

# drop geometry column and save as CSV for further analysis
gdf_join.drop(columns=["geometry"]).to_csv(OUT_CSV_BARRIOS, index=False)

print(f"saved venues with barrio/comuna to: {OUT_CSV_BARRIOS}")
print(f"total rows: {len(gdf_join)}")

# display sample of enriched data to check the merge was successful
gdf_join[["name", "barrio", "comuna"]].head(10)

saved venues with barrio/comuna to: data/processed/osm_naming_ba_porteno_with_barrios.csv
total rows: 4922


,name,barrio,comuna
0,Restaurant Museo Evita,Palermo,14.0
1,Voulez Bar,Palermo,14.0
2,Olivetti,Palermo,14.0
3,Bella Italia,Palermo,14.0
4,Las Asturias,Villa Luro,10.0
5,El Argentino,Villa Luro,10.0
6,bule-bar,Villa Luro,10.0
7,Avellino,Liniers,9.0
8,Café Tortoni,Monserrat,1.0
9,Dulce Hora,Versalles,10.0


## Step 3: Socioeconomic Data Preparation

**Goal:** Transform and merge comuna-level socioeconomic indicators for correlation analysis.

To test whether naming patterns correlate with neighborhood affluence or demographics, we need socioeconomic data that is available at the comuna level.

**Technical Approach:**
- Load demographic data and economic data as CSVs from the city's open data platform
- Clean column names and extract comuna numbers
- Parse some column values that will be useful for our analysis
- Merge demographic and economic data on comuna number

**Data Source:**
- Comunas en la web (https://www.estadisticaciudad.gob.ar/eyc/comunasenlaweb/)

**Output:** `comuna_demo_econ_merged.csv` — unified socioeconomic dataset at comuna level

In [33]:
# step 3: socioeconomic data preparation
# --------------------------------------------------------------------

import pandas as pd
import re

# ─────────────────────────────────────────────────────────────
# 1. clean comuna_demo (demographic data)
# ─────────────────────────────────────────────────────────────
df_demo = pd.read_csv('/Users/lara/Desktop/naming_a_city/data/comunas_data/comuna_demo.csv')

# drop metadata columns (indicador_grupo, Gráfico) and total column
df_demo_cleaned = df_demo.drop(columns=['indicador_grupo', 'Gráfico'], errors='ignore')

# set 'Indicador' column as row index
df_demo_cleaned = df_demo_cleaned.set_index('Indicador')

# transpose: comunas become rows, indicators become columns
df_demo_transposed = df_demo_cleaned.T

# extract comuna number (e.g., 'comuna_1' → 1)
def extract_comuna_number(s):
    if pd.isna(s):
        return None
    s = str(s).lower().strip()
    match = re.search(r'(\d+)', s)
    return int(match.group(1)) if match else None

# add comuna column
df_demo_transposed['comuna'] = df_demo_transposed.index.map(extract_comuna_number)
df_demo_transposed = df_demo_transposed.dropna(subset=['comuna'])
df_demo_transposed['comuna'] = df_demo_transposed['comuna'].astype(int)

# reset index to make comuna a regular column
df_demo_transposed = df_demo_transposed.reset_index(drop=True)

# rename columns to clean indicator names
demo_col_mapping = {
    'Población ': 'poblacion',
    'Superficie (km2)': 'superficie_km2',
    'Densidad (hab. por km2)': 'densidad_hab_km2',
    'Porcentaje de población en el total de la Ciudad': 'porc_poblacion_ciudad',
    'Porcentaje de superficie en el total de la Ciudad': 'porc_superficie_ciudad',
    'Cantidad de varones cada 100 mujeres': 'varones_cada_100_mujeres',
    'Edad promedio (años)': 'edad_promedio_anios',
    'Porcentaje de población de 0 - 14 años': 'porc_poblacion_0_14',
    'Porcentaje de población de 15 - 64 años': 'porc_poblacion_15_64',
    'Porcentaje de población de 65 años y más ': 'porc_65mas',
    'Porcentaje de población nacida en esta Ciudad': 'porc_nacidos_ciudad',
    'Tasa global de fecundidad (hijos/as por mujer)1 ': 'tasa_fecundidad',
    'Tasa de mortalidad infantil (por mil nacidos/as vivos/as) 1 ': 'tasa_mortalidad_infantil',
    'Tasa bruta de natalidad (por mil) (2023) ': 'tasa_natalidad',
    'Tasa bruta de mortalidad (por mil) (2023) ': 'tasa_mortalidad',
    'Tasa de crecimiento vegetativo (por mil) (2023)': 'tasa_crecimiento_vegetativo',
    'Porcentaje de mujeres de 14 años y más que no tuvieron hijos/as': 'porc_mujeres_sin_hijos',
    'Porcentaje de mujeres de 14 años y más que tuvieron 1 o 2 hijos/as ': 'porc_mujeres_1_2_hijos',
    'Porcentaje de mujeres de 14 años y más que tuvieron más de 2 hijos/as': 'porc_mujeres_mas_2_hijos',
    'Tamaño medio del hogar': 'tamano_medio_hogar',
    'Porcentaje de hogares unipersonales': 'porc_hogares_unipersonales',
    'Porcentaje de población de 14 años y más unida (en unión legal o consensual) ': 'porc_poblacion_unida',
    'Porcentaje de hogares familiares': 'porc_hogares_familiares',
    'Porcentaje de hogares no familiares': 'porc_hogares_no_familiares',
}
df_demo_transposed = df_demo_transposed.rename(columns=demo_col_mapping)

# save cleaned demographic data
df_demo_transposed.to_csv('/Users/lara/Desktop/naming_a_city/data/processed/comuna_demo_cleaned.csv', index=False)
print("saved cleaned demographic data to: data/processed/comuna_demo_cleaned.csv")

# ─────────────────────────────────────────────────────────────
# 2. clean comuna_econ (economic data)
# ─────────────────────────────────────────────────────────────
df_econ = pd.read_csv('/Users/lara/Desktop/naming_a_city/data/comunas_data/comuna_econ.csv')

# set 'Indicador' column as row names
df_econ_cleaned = df_econ.set_index('Indicador')

# transpose: comunas become rows, indicators become columns
df_econ_transposed = df_econ_cleaned.T

# extract comuna number
df_econ_transposed['comuna'] = df_econ_transposed.index.map(extract_comuna_number)
df_econ_transposed = df_econ_transposed.dropna(subset=['comuna'])
df_econ_transposed['comuna'] = df_econ_transposed['comuna'].astype(int)

# reset index
df_econ_transposed = df_econ_transposed.reset_index(drop=True)

# rename columns to clean snake_case names
econ_col_mapping = {
    'Promedio del Ingreso Per Cápita Familiar (IPCF) de los hogares (pesos) ': 'ipcf_promedio_pesos',
    'Porcentaje de hogares con ingresos inferiores a la Línea de Pobreza ': 'porc_hogares_bajo_lp',
    'Porcentaje de poblacion con ingresos inferiores a la Línea de Pobreza ': 'porc_poblacion_bajo_lp',
    'Tasa de actividad1 ': 'tasa_actividad',
    'Tasa de empleo1 ': 'tasa_empleo',
    'Tasa de desocupación1 ': 'tasa_desocupacion',
    'Tasa de desocupación de los varones ': 'tasa_desocupacion_varones',
    'Tasa de desocupación de las mujeres ': 'tasa_desocupacion_mujeres',
    'Tasa de subocupación horaria ': 'tasa_subocupacion',
    'Porcentaje de ocupados/as por cuenta propia': 'porc_cuenta_propia',
    'Porcentaje de asalariados/as sin descuento jubilatorio ': 'porc_asalariados_sin_jubilacion',
}

df_econ_transposed = df_econ_transposed.rename(columns=econ_col_mapping)

# save cleaned economic data
df_econ_transposed.to_csv('/Users/lara/Desktop/naming_a_city/data/processed/comuna_econ_cleaned.csv', index=False)
print("saved cleaned economic data to: data/processed/comuna_econ_cleaned.csv")

saved cleaned demographic data to: data/processed/comuna_demo_cleaned.csv
saved cleaned economic data to: data/processed/comuna_econ_cleaned.csv


In [34]:
# merge socioeconomic data: combine income and demographic data by comuna
# --------------------------------------------------------------------

# merge economic and demographic data on comuna number
df_merged = df_econ_transposed.merge(
    df_demo_transposed, 
    on="comuna", 
    how="outer"
)

# sort by comuna for easier inspection
df_merged = df_merged.sort_values("comuna").reset_index(drop=True)

# save combined socioeconomic dataset
df_merged.to_csv("data/processed/comuna_demo_econ_merged.csv", index=False)

# show sample data to check that the merge was successful
print(f"sample merged data:")
print(df_merged[['comuna', 'ipcf_promedio_pesos', 'edad_promedio_anios', 'porc_65mas']].head())


sample merged data:
Indicador  comuna ipcf_promedio_pesos edad_promedio_anios porc_65mas
0               1         ARS 694,384                37.8       14.9
1               2       ARS 1,046,451                42.5       21.1
2               3         ARS 685,561                39.2       16.8
3               4         ARS 573,468                35.0       13.4
4               5         ARS 859,444                40.3       18.1


In [35]:
# parse income column: transform "ARS 694,384" format in column ipcf_promedio_pesos to numeric 694384
# --------------------------------------------------------------------

df_merged["ipcf_promedio_pesos"] = (
    df_merged["ipcf_promedio_pesos"]
    .str.replace("ARS", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(int)
)

# save updated dataset
df_merged.to_csv("data/processed/comuna_demo_econ_merged.csv", index=False)

print("parsed income column and saved to comuna_demo_econ_merged.csv")


parsed income column and saved to comuna_demo_econ_merged.csv


In [36]:
# merge venues with socioeconomic data: attach comuna-level attributes to venues
# --------------------------------------------------------------------

# load venues with geographic attributes
df_places = pd.read_csv("data/processed/osm_naming_ba_porteno_with_barrios.csv")

# load combined socioeconomic data
df_comuna = pd.read_csv("data/processed/comuna_demo_econ_merged.csv")

# merge venues with socioeconomic data on comuna number
# left join ensures all venues are kept even if socioeconomic data is missing, just in case
df_merged_places = df_places.merge(df_comuna, on="comuna", how="left")

# save enriched dataset
df_merged_places.to_csv("data/processed/osm_naming_ba_porteno_only.csv", index=False)

print(f"merged venues with socioeconomic data: {len(df_merged_places)} rows")


merged venues with socioeconomic data: 4922 rows


## Step 4: Regex-based Cultural Motif Classification

**Goal:** Assign venues motif labels representing their cultural reference through regex.

**Technical Approach:**
1. **Name normalization:** Convert to lowercase, remove accents for consistent pattern matching
2. **Chain detection:** Flag businesses appearing >8 times. We don't want this venues to overpower the database.
3. **Regex classification:** Apply one of the cultural motifs defined using regex patterns

**Output:** `osm_naming_ba_porteno_only_enriched.csv` — venues with cultural motif labels

In [37]:
# step 4: cultural motif classification with regex patterns
# --------------------------------------------------------------------

import re, unicodedata, pandas as pd
from pathlib import Path

# load the pre-merged CSV with porteño OSM data + comuna info
df = pd.read_csv("data/processed/osm_naming_ba_porteno_only.csv")

# normalize names (lowercase + strip accents) for consistent pattern matching
def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c))

df["_name_norm"] = df["name"].apply(norm)

# detect chains by repeated normalized names
# we assume that if the same normalized name appears ≥8 times, it's a chain
name_counts = df["_name_norm"].value_counts()
CHAIN_THRESHOLD = 8

df["chain_count"] = df["_name_norm"].map(name_counts)
df["is_chain"] = df["chain_count"] >= CHAIN_THRESHOLD

# --------------------------------------------------------------------
# regex pattern definitions for cultural motifs
# --------------------------------------------------------------------

# 1) Religious / Devotional
RELIGIOSO = re.compile(r"""
\b(
  san|santa|santo|santisima|virgen|cristo|jesus|nazaret|belen|dios|
  cayetano|guadalupe|rosario|
  milagrit[oa]s?|milagrosa|misericordia|zion|
  paraiso|santos?|bendito
)\b
""", re.VERBOSE)

# 2) National Heroes and Patriotic Symbols
PROCERES = re.compile(r"""
\b(
  san\s+martin|belgrano|guemes|sarmiento|alvear|roca|moreno|mitre|brown|
  saavedra|rosas|rivadavia|pueyrredon|lavalle|dorrego|alberdi|viamonte|
  azcuenaga|french|beruti|laprida|anchorena|
  escarapela|patria|celeste\s+y\s+blanco|
  catedral|querandi
)\b
""", re.VERBOSE)

# 3) Neighboring Latin American Cultures
VECINOS = re.compile(r"""
\b(
  uruguay|uruguayo?s?|
  peru|peruan[ao]s?|inka|
  bolivia|bolivian[ao]s?|
  paraguay|paraguay[oa]s?|
  chile|chilen[oa]s?|
  brasil|brasilen[so]s?|carioca|rodizinho|
  colombia|colombian[ao]s?|
  andina|illimani|
  cusco|cuzco|guarani|
  arepas?|ceviche|pisco|chivito|chiviteria|
  bahia|lima|arequipa|cusquena|cusqueña
)\b
""", re.VERBOSE)

# 4) Italian Heritage
EUROPEA_IT = re.compile(r"""
\b(
  roma|milano|napoli|napoles|venezia|venecian[ao]|torino|sicilia|calabria|puglia|
  modena|parma|genova|bologna|firenze|fiorentina|italia|italian[ao]s?|italiana|
  trattoria|osteria|ristorante|enoteca|gelateria|pasticceria|ristretto|forchetta|
  ditali|broccolino|novecento|delizie|tentazioni|coraggio|prosciutto|freddo|
  rigoleto|rigoletto|tonno|persicco|burgio|vita|bellagamba|piccolo|
  vinci|diechi|vittorio|mamma|darsena|della|giovanni|
  sal(?:ute\s+)?garibaldi|damore|d'amore|d'onore|donore|capriccio|cocoliche|
  il|accademia(\s+della)?|cuore|piacere|nonna|d'oro|parolaccia|
  paradiso|amaretto|quotidi[ao]no?|giuseppe|tano
)\b
""", re.VERBOSE)

# 5) French Heritage
EUROPEA_FR = re.compile(r"""
\b(
  bistro|boulangerie|patisserie|fromagerie|brasserie|chez|creperie|brioche|
  brochette|vedette|pharmacie|lumiere|fondue|oui|moliere|brut|ohlala|
  madeleine|raclette|quartier|
  noire|suis|vivant|
  pain\s+et\s+vin|
  le|voulez|lharmonie|l'harmonie|jour|etre|petit|madame
)\b
""", re.VERBOSE)

# 6) Spanish Heritage
EUROPEA_ES = re.compile(r"""
\b(
  valenciana|espanol|catalunya|las\s+asturias|
  el\s+gallego|
  burladero|t'cocino|tcocino
)\b
""", re.VERBOSE)

# 7) Other European Heritage
EUROPEA_OTHER = re.compile(r"""
\b(
  andorra|montecarlo|griego|britanico|
  achtung\s+bar|hausbrot|kazan|eurocafe
)\b
""", re.VERBOSE)

# 8) Personal / Familiar
PERSONAL = re.compile(r"""
\b(
  lo\s+de|
  don\s+[a-záéíóúñ]+|
  doñ?a\s+[a-záéíóúñ]+|
  los\s+(primos|hermanos|mellizos)|
  las\s+hermanas|
  la\s+nona\b|la\s+abuela\b|
  (tio|tia|abuelo|manolo|jaimito)|
  casa|casero|
  (?:el|la)\s+tano\b|
  dona
)\b
""", re.VERBOSE)

# 9) Río de la Plata Nostalgia
NOSTALGIA = re.compile(
    r"\b(bodegon|pulperia|esquina|esquinita|tango|milonga|arrabal|bar\s+notable|el\s+federal|federal|el\s+viejo|la\s+vieja|cambalache|boliche|vinilo|boedo\s+antiguo|legado|farola|farolita)\b"
    r"|\b(18|19)[0-9]{2}\b"
)

# 10) Asian Heritage
ASIATICA = re.compile(r"""
\b(
  asia(\s+de\s+cuba)?|asiatic[oa]s?|asia\s+fusion|
  china|chino?s?|chinatown|tao|
  ying|yang|feng|
  saigon|yatoi|moshu|
  koh\s+lanta|
  dumplings?|
  asian|
  corea|korean|corean[ao]s?|
  japon|japoneses?|
  tokyo|osaka|kyoto|pekin|beijing|shanghai|bangkok|hanoi|taipei|
  sakura|ramen|sushi|sushibox|nigiri|maki|sashimi|tataki|
  teriyaki|gyoza|wok|tempura|udon|yakitori|yakisoba|okonomiyaki|
  izakaya|bento|kimchi|bibimbap|pho|bao|dim\s?sum|dumpling|
  fujisan|kanji|nobiru|norimoto|murasaki|mizuki|niji|
  kaffir|lai-?lai|yafuso|kunjip|nikkai|
  mikhuna
)\b
""", re.VERBOSE)

# 11) Jewish Heritage
JUDIA = re.compile(r"""
\b(
  benaim|moisha|moishe|moyshe|
  shalom|kosher|jab(?:ad|ot)|jabad|
  mishiguene|
  israeli|israelis?|
  shiva|
  judaic[ao]s?|hebre[oa]s?
)\b
""", re.VERBOSE)

# 12) Middle Eastern Heritage
MEDIO = re.compile(r"""
\b(
  kebab|shawarma|falafel|hummus|tabule|tabbouleh|labne?h?|fattoush|tahini|sumac|za+tar|
  manoush|manakish|lahma?cun|kofta|kofte|kibbeh|baklava|kunefe|knafeh|maamoul|halva|
  beirut|liban[oe]s|libano|damasco|aleppo|amman|jerusalen|jerusalem|istanbul|estambul|
  anatolia|sultan|otoman|
  arabe|arabes|turc[oa]s?|persa|iran[ií]?|levantin[oa]s?|sirio|siria|palestin[ao]s?|
  shisha|hookah|narghile|narguileh?|arguile|
  aladdin|aladin|ali\s+baba|alibaba|sahara|casablanca|marrake?sh|magh?reb|tanger|fe[sz]|
  habibi|jhabibi|
  medio\s+oriente|oriente|zein|rayan|sarkis
)\b
""", re.VERBOSE)

# 13) Argentine Folklore and Local Culture
FOLK_ARG = re.compile(r"""
\b(
  ceibo|ombu|jacaranda|lapacho|algarrobo|quebracho|hornero|yaguarete|zorzal|carpincho|guanaco|dorado|pejerrey|surubi|pacu|boga|
  mate|bombilla|yerba|criollo|criollos|gaucho|gauchos|gauchito|asado|empanadas?|parrilla|fogon|bodegon|almacen|choripan|
  laurel|laureles|laurelitos|minga|tradicional(?:es)?|bohemios|patagonia|
  regio|payuca|maradona|picaro|rancho|familia|veredita|desarmadero|malasangre|mulata|
  birra|birreria|querandi|colectivo|sanata|churros|morfar|negrin|gil|10s|conga|pibes?|amigos?|piba|saltenos|sudestada|
  tinglado|zeneize|ferneteria|humahuaca|gayola|fiera|bondi|bicisenda|banderin|sanjuanino|pancho|
  futbal|sifon|minutas|festin|ochava|milapizza|copetin|taragui|
  antojo|rabieta|
  futbol|cancha|barrabrava|hincha|boca|river|racing|independiente|san\s+lorenzo|huracan|argentinos|ferro|velez|quilmes|
  nueva\s+chicago|platense|lanus|banfield|tigre|
  pena|payada|pampa|zamba|chacarera|huella|patagonic[oa]|norten[oa]|cordobes|salten[oa]|mendocin[oa]|jujen[oa]|
  cholas?|cholita|cuarteta|cuartetas|erre|muchachos|zapi|
  rock(\s+nacional)?|sumo|soda\s+stereo|spinetta|cerati|charly|calamaro|la\s+renga|bersuit|pappo|viejas\s+locas|babasonicos|ratones\s+paranoicos|
  metegol|truco|barrio|villa|pibe|patio|boliche|carnaval|
  peron|evita|descamisado|justicialista|patria|nacion|argentin[ao]s?|malvinas|obelisco|abasto|
  al\s+margen|hora\s+libre|rey\s+de\s+copas|
  \bche\b|porten[oa]s?|amores\s+portenos?
)\b
""", re.VERBOSE)

# 14) Community Names / Union / Popular
NOMBRES_COMUNITARIOS = re.compile(r"""
\b(
  encuentro(\s+\w+)?|
  la\s+union|
  la\s+unión|
  union\s+\w+|
  unión\s+\w+|
  la\s+esperanza(\s+de\s+\w+)?|
  la\s+victoria|
  gran\s+victoria|
  el\s+popular(\s+de\s+\w+)?|
  la\s+popular(\s+de\s+\w+)?|
  el\s+podio|
  el\s+refuerzo
)\b
""", re.VERBOSE | re.IGNORECASE)

# 15) Animals and Nature (Neutral)
ANIMALES_NAT = re.compile(r"""
\b(
  galg[oa]s?|carpincho|manzanitas?|mosquito|molino|molinera|caballo|caballito|
  mono|monos|gato|gata|gatos|perro|perra|perros|can|caniche|felino|felinos|
  leon|leona|tigre|tigres|pantera|lobo|lobos|zorro|zorros|zorrino|zorrinos|
  oso|osos|ciervo|ciervos|liebre|conejo|conejos|burro|burros|asno|asnos|
  oveja|ovejas|cabra|cabras|cerdo|cerdos|chancho|chanchos|vaca|vacas|toro|toros|yegua|potro|
  pollo|gallo|gallina|gallinas|paloma|palomas|canario|canarios|gorrion|gorriones|colibri|colibries|
  loro|loros|aguila|aguilas|halcon|halcones|buho|lechuza|cuervo|cuervos|pato|patos|ganso|gansos|
  pez|peces|delfin|delfines|tiburon|tiburones|ballena|ballenas|mariposa|mariposas|abeja|abejas|hormiga|hormigas|arana|aranas|
  salmon|cisne|pinguino|chanchitos|fauna|muu|
  arbol|arboleda|bosque|selva|rio|parana|laguna|lago|mar|oceano|playa|arena|costa|bahia|catarata|cataratas|
  viento|aire|cielo|nube|nubes|sol|luna|estrella|estrellas|montana|montanas|sierra|sierras|cordillera|valle|valles|
  pradera|campo|jardin|jardines|prado|prados|cesped|cespedes|pasto|grama|hierba|hoja|hojas|flor|flores|loto|tallo|raiz|raices|roca|rocas|piedra|piedras|
  solar|magnolia|trebol|petunias|cosecha|artemisia|oasis
)\b
""", re.VERBOSE)

# 16) Literary / Intellectual / Artistic
LITERARIO = re.compile(r"""
\b(
  borges|cortazar|rayuela|ateneo|eterna\s+cadencia|sabato|galeano|bioy|pizarnik|arlt|marechal|
  kafka|orwell|onetti|benedetti|cervantes|quijote|ulises|poesia|lorca|socrates|tita\s+merello|
  haroldo\s+conti|hilario\s+ascasubi|leopoldo\s+lugones|
  vivaldi|kandinsky|monet|dali|gaudi|
  tunel|arte|guitarrita|guitarra|trova|
  artistas?|artista|
  moby\s+dick|
  mil\s+y\s+una\s+noches|mil\s+y\s+una\s+hijas|
  4ta\s+pared|
  osvaldo\s+soriano|rafael\s+obligado|rodolfo\s+walsh|silvina\s+ocampo|
  bonnie\s*&\s*clyde
)\b
""", re.VERBOSE)

# 17) Anglophone
ANGLOPHONE = re.compile(r"""
(
  ^the\s
|
  \'s\b
|
  \b(starbucks|mcdonalds|big\s+food)\b
|
  \b(let\s+it|i\s+am)\b
|
  \b(
    coffee|house|shop|grill|burger|brew|studio|design|market|pride|velvet|
    notorious|green\s+life|open|kentucky|family|lately|subway|company|eat|
    relax|hard\s+rock\s+cafe|boston|new|uptown|good|steaks|pretty|mood|tea|
    crowlers|juice|walrus|mellow|friends|george|however|makers|green|garden|
    tipsy|superfish|happening|monday|big|delicious|place|stars|hills|import|
    pepper|fame|sweetly|flat&white|diggs|georgie?s?
  )\b
)
""", re.VERBOSE)

# 18) New Age
NEW_AGE = re.compile(r"""
\b(
  vibra|vibra\s+positivo|positivo|mandala|zen|chakra|chakra[s]?|
  aura|alma|alquimia|karma|yin|yang|feng\s+shui|
  despertar|despierta|renacer|renacimiento|
  arcano|tarot|esoteri[co]s?|esoterica|
  solo\s+por\s+hoy|un\s+dia\s+a\s+la\s+vez|
  milagro|milagros|
  buena\s+onda|onda\s+positiva|
  conciencia|consciente|mindfulness|
  sanacion|sanar|sanando
)\b
""", re.VERBOSE)

# priority order for classification
# names are tested against patterns in this order, so each name receives a single, most-relevant label
PRIORITY = [
    ("Anglophone", ANGLOPHONE),
    ("Religioso / devocional", RELIGIOSO),
    ("Próceres nacionales y símbolos patrios", PROCERES),
    ("Culturas latinoamericanas vecinas", VECINOS),
    ("Herencia europea (italiana)", EUROPEA_IT),
    ("Herencia europea (francesa)", EUROPEA_FR),
    ("Herencia europea (española)", EUROPEA_ES),
    ("Herencia europea (otra)", EUROPEA_OTHER),
    ("Herencia judía", JUDIA),
    ("Herencia de Medio Oriente", MEDIO),
    ("Herencia asiática", ASIATICA),
    ("Folklore y cultura local argentina", FOLK_ARG),
    ("Personal / familiar", PERSONAL),
    ("Nostalgia rioplatense", NOSTALGIA),
    ("Literario / intelectual / artístico", LITERARIO),
    ("New Age / self help", NEW_AGE),
    ("Nombres comunitarios / unión / popular", NOMBRES_COMUNITARIOS),
    ("Animales y naturaleza (neutral)", ANIMALES_NAT),
]

# classification helper: assign a single semantic label per name
def classify(name: str) -> str:
    """
    classify venue name by testing against regex patterns in priority order.
    returns the label for the first matching pattern, or generic if no match.
    """
    s = norm(name)
    
    # try each pattern in priority order, return the label for the first match
    for label, pat in PRIORITY:
        if pat.search(s):
            return label
    
    # if no explicit cultural pattern was found, return generic
    return "Sin motivo claro / nombre genérico"

# apply classification to all rows
df["regex_label"] = df["name"].apply(classify)

# save enriched dataset
# this file now contains:
#   - original OSM fields (name, key, value, coordinates, etc.)
#   - barrio/comuna info (from spatial join)
#   - socioeconomic data (from comuna merge)
#   - chain metadata (chain_count, is_chain)
#   - regex-based cultural label (regex_label)
out_path = Path("data/processed/osm_naming_ba_porteno_only_enriched.csv")
df.to_csv(out_path, index=False)

# translated labels for display
label_translations = {
    "Religioso / devocional": "Religious / Devotional",
    "Próceres nacionales y símbolos patrios": "National Heroes and Patriotic Symbols",
    "Culturas latinoamericanas vecinas": "Neighboring Latin American Cultures",
    "Herencia europea (italiana)": "Italian Heritage",
    "Herencia europea (francesa)": "French Heritage",
    "Herencia europea (española)": "Spanish Heritage",
    "Herencia europea (otra)": "Other European Heritage",
    "Herencia judía": "Jewish Heritage",
    "New Age / self help": "New Age",
    "Personal / familiar": "Personal / Familiar",
    "Nostalgia rioplatense": "Río de la Plata Nostalgia",
    "Herencia asiática": "Asian Heritage",
    "Herencia de Medio Oriente": "Middle Eastern Heritage",
    "Folklore y cultura local argentina": "Argentine Folklore and Local Culture",
    "Nombres comunitarios / unión / popular": "Community Names",
    "Animales y naturaleza (neutral)": "Animals and Nature",
    "Literario / intelectual / artístico": "Artistic / Intellectual",
    "Anglophone": "Anglophone",
    "Sin motivo claro / nombre genérico": "No Clear Motive / Generic Name",
}

# print category counts with Spanish + English labels
counts = df["regex_label"].value_counts(dropna=False)
print("\n=== category summary (Spanish → English) ===")
for label, count in counts.items():
    english = label_translations.get(label, "")
    print(f"{label:<45} → {english:<50} | {count}")

print(f"\nguardado: {out_path}")



=== category summary (Spanish → English) ===
Sin motivo claro / nombre genérico            → No Clear Motive / Generic Name                     | 3350
Anglophone                                    → Anglophone                                         | 341
Folklore y cultura local argentina            → Argentine Folklore and Local Culture               | 298
Herencia europea (italiana)                   → Italian Heritage                                   | 129
Personal / familiar                           → Personal / Familiar                                | 128
Animales y naturaleza (neutral)               → Animals and Nature                                 | 126
Herencia europea (francesa)                   → French Heritage                                    | 113
Religioso / devocional                        → Religious / Devotional                             | 102
Herencia asiática                             → Asian Heritage                                     | 81
Nostalgia

In [38]:
# export unclassified names for LLM + manual review

import pandas as pd

# load enriched dataset
df = pd.read_csv("data/processed/osm_naming_ba_porteno_only_enriched.csv")

# filter to unclassified names only
unclassified = df[df["regex_label"] == "Sin motivo claro / nombre genérico"]

# keep only essential columns
unclassified = unclassified[["name", "regex_label"]].sort_values("name")

# save for LLM + manual review
unclassified.to_csv(
    "data/processed/unclassified.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"saved unclassified names to data/processed/unclassified.csv")
print(f"total unclassified: {len(unclassified)}")


saved unclassified names to data/processed/unclassified.csv
total unclassified: 3350


## Step 5: LLM Overrides

**Goal:** Apply specific corrections for names that regex cannot capture.

**Methodology:**
- Review unclassified venues using an LLM
- Apply specific overrides

**Output:** `osm_naming_ba_porteno_only_enriched_with_manual_overrides.csv`

In [39]:
# LLM-based manual overrides for specific names
# -------------------------------------------------------------
manual_overrides = {
    # ─────────────────────────────
    # Community / union / popular names
    # ─────────────────────────────
    "Asamblea Plaza": "Nombres comunitarios / unión / popular",
    "La Blanquita (Nacional y Popular)": "Nombres comunitarios / unión / popular",
    "Juntos Hasta El Final": "Nombres comunitarios / unión / popular",
    "La Tribu": "Nombres comunitarios / unión / popular",
    "La Voluntad": "Nombres comunitarios / unión / popular",
    "Tribu": "Nombres comunitarios / unión / popular",
    "Trybe": "Nombres comunitarios / unión / popular",
    "Utopia": "Nombres comunitarios / unión / popular",
    "Vereda Adentro": "Nombres comunitarios / unión / popular",
    "UOCRA Cultura": "Nombres comunitarios / unión / popular",

    # ─────────────────────────────
    # Jewish heritage
    # ─────────────────────────────
    "Eben Ezer": "Herencia judía",   

    # ─────────────────────────────
    # Religious / devotional
    # ─────────────────────────────
    "4 Angelitos": "Religioso / devocional",
    "All Saints Café": "Religioso / devocional",
    "Cafe Altares": "Religioso / devocional",
    "Café Altares": "Religioso / devocional",
    "Editorial Peniel": "Religioso / devocional",
    "El Maná": "Religioso / devocional",
    "Horus": "Religioso / devocional",
    "La Madonnina": "Religioso / devocional",
    "La Piedad": "Religioso / devocional",
    "La Pilarica": "Religioso / devocional",
    "Las Nazarenas": "Religioso / devocional",
    "Librería Cristiana": "Religioso / devocional",
    "Librería Marista": "Religioso / devocional",
    "Libros El Arca de Noé": "Religioso / devocional",
    "Los 3 Reyes Magos": "Religioso / devocional",
    "Lourdes": "Religioso / devocional",
    "Maná": "Religioso / devocional",
    "María": "Religioso / devocional",
    "María de Bambi": "Religioso / devocional",
    "Mi esperanza": "Religioso / devocional",
    "Nuestra América": "Religioso / devocional",
    "Panadería y Confitería Lourdes": "Religioso / devocional",
    "Ragnarok": "Religioso / devocional",
    "Relicioso": "Religioso / devocional",
    "Sacro": "Religioso / devocional",
    "Stella Maris": "Religioso / devocional",
    "Temple": "Religioso / devocional",
    "Temple Craft Soho": "Religioso / devocional",
    "VALHALLA Bar Vikingo": "Religioso / devocional",
    "Taberna Odín": "Religioso / devocional",
    "Vrindavan": "Religioso / devocional",

    # ─────────────────────────────
    # Personal / familiar
    # ─────────────────────────────
    "Abuela Pan": "Personal / familiar",
    "El Pobre Luis": "Personal / familiar",
    "El Rebenque de Omar": "Personal / familiar",
    "El Rincón de Pablo": "Personal / familiar",
    "El Tata": "Personal / familiar",
    "El Gordó Vegano": "Personal / familiar",
    "Lucio": "Personal / familiar",
    "Lupita": "Personal / familiar",
    "Mago": "Personal / familiar",
    "María Mia": "Personal / familiar",
    "Maricel": "Personal / familiar",
    "Martina": "Personal / familiar",
    "Matilda": "Personal / familiar",
    "Mimí": "Personal / familiar",
    "Miriam": "Personal / familiar",
    "Mr. Pedro": "Personal / familiar",
    "Serafin": "Personal / familiar",
    "Simon": "Personal / familiar",
    "Simón": "Personal / familiar",
    "Simona": "Personal / familiar",
    "Simona Café": "Personal / familiar",
    "Simone": "Personal / familiar",
    "Simonetta": "Personal / familiar",
    "Simão": "Personal / familiar",
    "Solomia": "Personal / familiar",
    "Sophia": "Personal / familiar",
    "Sr. Telmo": "Personal / familiar",
    "Thiago": "Personal / familiar",
    "Thomas": "Personal / familiar",
    "Tino": "Personal / familiar",
    "Tobias": "Personal / familiar",
    "Tulio": "Personal / familiar",
    "Torcuato & Regina": "Personal / familiar",
    "Vargas": "Personal / familiar",
    "Venancio": "Personal / familiar",
    "Vera": "Personal / familiar",
    "Verico": "Personal / familiar",
    "Vicente": "Personal / familiar",
    "Vickin": "Personal / familiar",
    "Villalobos": "Personal / familiar",
    "Yenny": "Personal / familiar",

    # ─────────────────────────────
    # Folklore and local Argentine culture
    # ─────────────────────────────
    "Aconcagua": "Folklore y cultura local argentina",
    "Almagro": "Folklore y cultura local argentina",
    "Alsina": "Folklore y cultura local argentina",
    "Anchoita": "Folklore y cultura local argentina",
    "Anchoita Panederia": "Folklore y cultura local argentina",
    "Aramburu": "Folklore y cultura local argentina",
    "Aramburu Restó": "Folklore y cultura local argentina",
    "Aramburu": "Folklore y cultura local argentina",
    "Aramburu Restó": "Folklore y cultura local argentina",
    "Asador Criolio": "Folklore y cultura local argentina",
    "Bar Argerich": "Folklore y cultura local argentina",
    "Bar Tucumán": "Folklore y cultura local argentina",
    "Benavidez": "Folklore y cultura local argentina",
    "Boedo XXI": "Folklore y cultura local argentina",
    "Bombonera Sandwichería": "Folklore y cultura local argentina",
    "Buenos Aires Verde": "Folklore y cultura local argentina",
    "Buenos Ayres Pizza": "Folklore y cultura local argentina",
    "Cabaña Tuyu": "Folklore y cultura local argentina",
    "Canillita": "Folklore y cultura local argentina",
    "Chaco bar": "Folklore y cultura local argentina",
    "Chacras de Ushuaia": "Folklore y cultura local argentina",
    "Club Milanesa": "Folklore y cultura local argentina",
    "Club de la Milanesa": "Folklore y cultura local argentina",
    "Corazón de mi tierra": "Folklore y cultura local argentina",
    "Costumbres Criollas": "Folklore y cultura local argentina",
    "El Bagual": "Folklore y cultura local argentina",
    "El Chaqueño": "Folklore y cultura local argentina",
    "El Horno del Norte": "Folklore y cultura local argentina",
    "El Obrero": "Folklore y cultura local argentina",
    "El Rey de la Bondiola": "Folklore y cultura local argentina",
    "Enfundá la mandolina": "Folklore y cultura local argentina",
    "Estación Caminito": "Folklore y cultura local argentina",
    "Gordo Chanta": "Folklore y cultura local argentina",
    "Gualicho": "Folklore y cultura local argentina",
    "Itatí": "Folklore y cultura local argentina",
    "La Aguada": "Folklore y cultura local argentina",
    "La casona del nono": "Folklore y cultura local argentina",
    "La Entrerriana": "Folklore y cultura local argentina",
    "La Estancia": "Folklore y cultura local argentina",
    "La Gran Nuñez": "Folklore y cultura local argentina",
    "La Palermo": "Folklore y cultura local argentina",
    "La Parri": "Folklore y cultura local argentina",
    "La Pascana": "Folklore y cultura local argentina",
    "La Porteñita": "Folklore y cultura local argentina",
    "La Posta": "Folklore y cultura local argentina",
    "La Posta de Los Tucu": "Folklore y cultura local argentina",
    "La Querencia": "Folklore y cultura local argentina",
    "La Recova de Soldati": "Folklore y cultura local argentina",
    "La Super Boquense": "Folklore y cultura local argentina",
    "La Tranquera": "Folklore y cultura local argentina",
    "La muzzicata": "Folklore y cultura local argentina",
    "Lezama": "Folklore y cultura local argentina",
    "Lalo de Buenos Aires": "Folklore y cultura local argentina",
    "Las Breñas": "Folklore y cultura local argentina",
    "Las Lenas": "Folklore y cultura local argentina",
    "Las Leñas": "Folklore y cultura local argentina",
    "Llao Llao": "Folklore y cultura local argentina",
    "Lola Mora": "Folklore y cultura local argentina",
    "Los Corrales": "Folklore y cultura local argentina",
    "Los Nonos de Barracas": "Folklore y cultura local argentina",
    "Los Tulumbanos": "Folklore y cultura local argentina",
    "Los Patricios": "Folklore y cultura local argentina",
    "Madero": "Folklore y cultura local argentina",
    "Muley Muley": "Folklore y cultura local argentina",
    "Nativo": "Folklore y cultura local argentina",
    "Natural Buenos Aires": "Folklore y cultura local argentina",
    "Paladar Negro": "Folklore y cultura local argentina",
    "Pampero": "Folklore y cultura local argentina",
    "Patagones": "Folklore y cultura local argentina",
    "Posada de Virreyes": "Folklore y cultura local argentina",
    "Pucará": "Folklore y cultura local argentina",
    "Sabores de Urquiza": "Folklore y cultura local argentina",
    "Sabores de mi Tierra": "Folklore y cultura local argentina",
    "Santafesino": "Folklore y cultura local argentina",
    "Saujil": "Folklore y cultura local argentina",
    "Telmo Deli": "Folklore y cultura local argentina",
    "Teodelina": "Folklore y cultura local argentina",
    "Tierra de Indios": "Folklore y cultura local argentina",
    "Tobas Café": "Folklore y cultura local argentina",
    "UCO": "Folklore y cultura local argentina",
    "UOCRA Cultura": "Folklore y cultura local argentina",
    "Vereda Adentro": "Folklore y cultura local argentina",
    "Verm y Cheli": "Folklore y cultura local argentina",
    "Viejo Mundo": "Folklore y cultura local argentina",
    "Viejo Alsina": "Folklore y cultura local argentina",
    "Viejo Gómez": "Folklore y cultura local argentina",
    "Viejo Agump": "Folklore y cultura local argentina",
    "Viva Lugano": "Folklore y cultura local argentina",
    "Varela Varelita": "Folklore y cultura local argentina",
    "la Aguada": "Folklore y cultura local argentina",
    "la casona del nono": "Folklore y cultura local argentina",
    "el Nacional": "Folklore y cultura local argentina",
    "centro basque": "Folklore y cultura local argentina",
    "Ñato": "Folklore y cultura local argentina",

    # ─────────────────────────────
    # Rio de la Plata nostalgia
    # ─────────────────────────────
    "Bar El Coloñal": "Nostalgia rioplatense",
    "Bar El Destello": "Nostalgia rioplatense",
    "Bar El Faro": "Nostalgia rioplatense",
    "Bar Seddon": "Nostalgia rioplatense",
    "Bar Sur": "Nostalgia rioplatense",
    "Bar de Cao": "Nostalgia rioplatense",
    "Bar y Vermú La Fuerza": "Nostalgia rioplatense",
    "Café de Garcia": "Nostalgia rioplatense",
    "Café de los Angelitos": "Nostalgia rioplatense",
    "Café La Biela": "Nostalgia rioplatense",
    "Café Margot": "Nostalgia rioplatense",
    "Café Rivas": "Nostalgia rioplatense",
    "El Hipopótamo": "Nostalgia rioplatense",
    "El Progreso": "Nostalgia rioplatense",
    "El Preferido de Palermo": "Nostalgia rioplatense",
    "El tropezón": "Nostalgia rioplatense",
    "Estacion Caminito": "Nostalgia rioplatense",
    "Gardel": "Nostalgia rioplatense",
    "Güerrín": "Nostalgia rioplatense",
    "La Academia": "Nostalgia rioplatense",
    "La Viruta": "Nostalgia rioplatense",
    "La colonial": "Nostalgia rioplatense",
    "Las Violetas": "Nostalgia rioplatense",
    "Los 36 Billares": "Nostalgia rioplatense",
    "Los Inmortales": "Nostalgia rioplatense",
    "Los Inmortales - Corrientes": "Nostalgia rioplatense",
    "Pizza Angelín": "Nostalgia rioplatense",
    "Plaza Asturias": "Nostalgia rioplatense",
    "Plaza Mayor": "Nostalgia rioplatense",
    "Plaza serrano": "Nostalgia rioplatense",
    "Tabaris": "Nostalgia rioplatense",

    # ─────────────────────────────
    # National heroes and patriotic symbols
    # ─────────────────────────────
    "Cabildo de Buenos Aires": "Próceres nacionales y símbolos patrios",

    # ─────────────────────────────
    # Latin American neighbor cultures
    # ─────────────────────────────
    "Ayacucho Hotel Bar": "Culturas latinoamericanas vecinas",
    "Babalú": "Culturas latinoamericanas vecinas",
    "Bar Ecuador": "Culturas latinoamericanas vecinas",
    "Bogotá": "Culturas latinoamericanas vecinas",
    "Caracas Bar": "Culturas latinoamericanas vecinas",
    "Chalaca": "Culturas latinoamericanas vecinas",
    "Chan Chan": "Culturas latinoamericanas vecinas",
    "Chiperia": "Culturas latinoamericanas vecinas",
    "Cinépolis": "Culturas latinoamericanas vecinas",
    "Cinépolis Recoleta": "Culturas latinoamericanas vecinas",
    "Cohiba": "Culturas latinoamericanas vecinas",
    "Copacabana": "Culturas latinoamericanas vecinas",
    "Cuba Mía": "Culturas latinoamericanas vecinas",
    "Cumana": "Culturas latinoamericanas vecinas",
    "Cumaná": "Culturas latinoamericanas vecinas",
    "Darién": "Culturas latinoamericanas vecinas",
    "Frida": "Culturas latinoamericanas vecinas",
    "Juan Valdez Café": "Culturas latinoamericanas vecinas",
    "Juan Valdéz Café": "Culturas latinoamericanas vecinas",
    "La Atenas de Cuba": "Culturas latinoamericanas vecinas",
    "La Jalapeña": "Culturas latinoamericanas vecinas",
    "La Paceña": "Culturas latinoamericanas vecinas",
    "La Paz": "Culturas latinoamericanas vecinas",
    "La Puerto Rico": "Culturas latinoamericanas vecinas",
    "Lab Sucre": "Culturas latinoamericanas vecinas",
    "Larense": "Culturas latinoamericanas vecinas",
    "Los Orientales": "Culturas latinoamericanas vecinas",
    "Medio y Medio": "Culturas latinoamericanas vecinas",
    "Mezcal": "Culturas latinoamericanas vecinas",
    "Nuestra América": "Culturas latinoamericanas vecinas",
    "Ouro Preto": "Culturas latinoamericanas vecinas",
    "Playas del Carmen": "Culturas latinoamericanas vecinas",
    "Primavera Trujillana": "Culturas latinoamericanas vecinas",
    "primavera Trujillana": "Culturas latinoamericanas vecinas",
    "Rapanui": "Culturas latinoamericanas vecinas",
    "Rest Apu": "Culturas latinoamericanas vecinas",
    "Rodizio": "Culturas latinoamericanas vecinas",
    "Romario": "Culturas latinoamericanas vecinas",
    "Romario Berutti": "Culturas latinoamericanas vecinas",
    "Sabor Andino": "Culturas latinoamericanas vecinas",
    "Sabores de mi Tierra": "Culturas latinoamericanas vecinas",
    "Salteñas Oruro": "Culturas latinoamericanas vecinas",
    "Sampa": "Culturas latinoamericanas vecinas",
    "Sando de América": "Culturas latinoamericanas vecinas",
    "Sao Paolo": "Culturas latinoamericanas vecinas",
    "Sucre": "Culturas latinoamericanas vecinas",
    "Sucre Restorán": "Culturas latinoamericanas vecinas",
    "Sur Latino": "Culturas latinoamericanas vecinas",
    "Taco Box": "Culturas latinoamericanas vecinas",
    "Taco box": "Culturas latinoamericanas vecinas",
    "Taco God": "Culturas latinoamericanas vecinas",
    "Taco N Todo": "Culturas latinoamericanas vecinas",
    "Tacos Express": "Culturas latinoamericanas vecinas",
    "TaqueGusta": "Culturas latinoamericanas vecinas",
    "Tanta": "Culturas latinoamericanas vecinas",
    "Taquería Diaz": "Culturas latinoamericanas vecinas",
    "Tequila": "Culturas latinoamericanas vecinas",
    "Tierra de Héroes": "Culturas latinoamericanas vecinas",
    "Tintico": "Culturas latinoamericanas vecinas",
    "Todo Mundo": "Culturas latinoamericanas vecinas",
    "Ulúa": "Culturas latinoamericanas vecinas",
    "Venezolano": "Culturas latinoamericanas vecinas",
    "Xalapa": "Culturas latinoamericanas vecinas",
    "Y que sabor!": "Culturas latinoamericanas vecinas",
    "Ya Cabron": "Culturas latinoamericanas vecinas",
    "i Latina": "Culturas latinoamericanas vecinas",

    # ─────────────────────────────
    # Italian heritage
    # ─────────────────────────────
    "Al Bacio": "Herencia europea (italiana)",
    "Al Taglio": "Herencia europea (italiana)",
    "Alessandro": "Herencia europea (italiana)",
    "Alimentari": "Herencia europea (italiana)",
    "Allungata": "Herencia europea (italiana)",
    "Angolo": "Herencia europea (italiana)",
    "Antica Berna": "Herencia europea (italiana)",
    "Antonietta": "Herencia europea (italiana)",
    "Antonino": "Herencia europea (italiana)",
    "Aquilanti Libros Antigüos y Modernos": "Herencia europea (italiana)",
    "Aromi": "Herencia europea (italiana)",
    "Avellino": "Herencia europea (italiana)",
    "Basilico": "Herencia europea (italiana)",
    "Bellaria": "Herencia europea (italiana)",
    "Belvedere Panaderia": "Herencia europea (italiana)",
    "Benedetta": "Herencia europea (italiana)",
    "Bianca": "Herencia europea (italiana)",
    "Bisatti": "Herencia europea (italiana)",
    "Biscottini": "Herencia europea (italiana)",
    "Bollito": "Herencia europea (italiana)",
    "Brera": "Herencia europea (italiana)",
    "Bricco": "Herencia europea (italiana)",
    "Brucia": "Herencia europea (italiana)",
    "Bruni": "Herencia europea (italiana)",
    "Brusco café & Bar": "Herencia europea (italiana)",
    "Cafe Boccazzi": "Herencia europea (italiana)",
    "Caffe Los Molinos": "Herencia europea (italiana)",
    "Caffe Nascosto": "Herencia europea (italiana)",
    "Caffe Vergnano": "Herencia europea (italiana)",
    "Caffé Doge": "Herencia europea (italiana)",
    "Caffé del Doge": "Herencia europea (italiana)",
    "Cafiolo": "Herencia europea (italiana)",
    "Café Matteo": "Herencia europea (italiana)",
    "Café Tortoni": "Herencia europea (italiana)",
    "Café del Doge": "Herencia europea (italiana)",
    "Capricci": "Herencia europea (italiana)",
    "Carbonetti": "Herencia europea (italiana)",
    "Carletto": "Herencia europea (italiana)",
    "Casanova": "Herencia europea (italiana)",
    "Ciao!": "Herencia europea (italiana)",
    "Cilento": "Herencia europea (italiana)",
    "Circolo Massimo": "Herencia europea (italiana)",
    "Comacchio": "Herencia europea (italiana)",
    "Cosi mi Piace": "Herencia europea (italiana)",
    "Costanzo": "Herencia europea (italiana)",
    "Cremolatti": "Herencia europea (italiana)",
    "D'Accordo": "Herencia europea (italiana)",
    "Da Mingo": "Herencia europea (italiana)",
    "Dante en Verona": "Herencia europea (italiana)",
    "Faricci": "Herencia europea (italiana)",
    "Farinelli": "Herencia europea (italiana)",
    "Filippa": "Herencia europea (italiana)",
    "Giusseppe Vicenti": "Herencia europea (italiana)",
    "Isla de Capri": "Herencia europea (italiana)",
    "Ispica": "Herencia europea (italiana)",
    "La Bistecca": "Herencia europea (italiana)",
    "La Camorra": "Herencia europea (italiana)",
    "La Faina": "Herencia europea (italiana)",
    "La Fonte De Oro": "Herencia europea (italiana)",
    "La Fuccinalta": "Herencia europea (italiana)",
    "La Gioconda": "Herencia europea (italiana)",
    "La Madonnina": "Herencia europea (italiana)",
    "La Napolitana": "Herencia europea (italiana)",
    "La Panotteca": "Herencia europea (italiana)",
    "La Parolacha": "Herencia europea (italiana)",
    "La Pesceria": "Herencia europea (italiana)",
    "La Piazza": "Herencia europea (italiana)",
    "La Porta Nera": "Herencia europea (italiana)",
    "La Pécora Nera": "Herencia europea (italiana)",
    "La Reggina": "Herencia europea (italiana)",
    "La Rossi Maniera": "Herencia europea (italiana)",
    "La Sorellina Pizza Bar": "Herencia europea (italiana)",
    "La Stampa": "Herencia europea (italiana)",
    "La Strega": "Herencia europea (italiana)",
    "Lievito madre": "Herencia europea (italiana)",
    "Ligure": "Herencia europea (italiana)",
    "Londra": "Herencia europea (italiana)",
    "Los Luiggis II": "Herencia europea (italiana)",
    "Lucca Heladería Boutique": "Herencia europea (italiana)",
    "Macarella": "Herencia europea (italiana)",
    "Macarro": "Herencia europea (italiana)",
    "Mancini": "Herencia europea (italiana)",
    "Mangiare": "Herencia europea (italiana)",
    "Mangini": "Herencia europea (italiana)",
    "Martinelli": "Herencia europea (italiana)",
    "Martinelli Café": "Herencia europea (italiana)",
    "Mostachole": "Herencia europea (italiana)",
    "Mozarella": "Herencia europea (italiana)",
    "Napulé": "Herencia europea (italiana)",
    "Olivetti": "Herencia europea (italiana)",
    "Palermo Vecchio": "Herencia europea (italiana)",
    "Panuccio": "Herencia europea (italiana)",
    "Parecchio": "Herencia europea (italiana)",
    "Pascale": "Herencia europea (italiana)",
    "Pastas Frescas Artesanales": "Herencia europea (italiana)",
    "Pastas Savori": "Herencia europea (italiana)",
    "Peppino": "Herencia europea (italiana)",
    "Per-Noi Cafe": "Herencia europea (italiana)",
    "Pertutti": "Herencia europea (italiana)",
    "Perutti": "Herencia europea (italiana)",
    "Piani": "Herencia europea (italiana)",
    "Pía Dolce": "Herencia europea (italiana)",
    "Piedimonte": "Herencia europea (italiana)",
    "Piedimonte Full": "Herencia europea (italiana)",
    "Pierino": "Herencia europea (italiana)",
    "Pinuccio": "Herencia europea (italiana)",
    "Piola": "Herencia europea (italiana)",
    "Pippo": "Herencia europea (italiana)",
    "Pizzería Tagliata": "Herencia europea (italiana)",
    "Pizzicato": "Herencia europea (italiana)",
    "Positano": "Herencia europea (italiana)",
    "Pranzo": "Herencia europea (italiana)",
    "Priamo": "Herencia europea (italiana)",
    "Proviamo": "Herencia europea (italiana)",
    "Rimini": "Herencia europea (italiana)",
    "Ritorno": "Herencia europea (italiana)",
    "Rondinella": "Herencia europea (italiana)",
    "Saltanapoli": "Herencia europea (italiana)",
    "Sesto": "Herencia europea (italiana)",
    "Siamo Nel Forno": "Herencia europea (italiana)",
    "Sopressata": "Herencia europea (italiana)",
    "Sorrento": "Herencia europea (italiana)",
    "Sotovoce cafe": "Herencia europea (italiana)",
    "Spazzio": "Herencia europea (italiana)",
    "Spiedo": "Herencia europea (italiana)",
    "Spiga": "Herencia europea (italiana)",
    "Stracqua": "Herencia europea (italiana)",
    "Tallarica": "Herencia europea (italiana)",
    "Tanoira": "Herencia europea (italiana)",
    "Teglia": "Herencia europea (italiana)",
    "Terranova": "Herencia europea (italiana)",
    "Tiberio": "Herencia europea (italiana)",
    "Tirrenia": "Herencia europea (italiana)",
    "Tomasso": "Herencia europea (italiana)",
    "Traiano": "Herencia europea (italiana)",
    "Trionfale": "Herencia europea (italiana)",
    "Tullo Pane": "Herencia europea (italiana)",
    "Tutto Pane": "Herencia europea (italiana)",
    "Tutto Panne": "Herencia europea (italiana)",
    "Undici": "Herencia europea (italiana)",
    "Vaffanculo": "Herencia europea (italiana)",
    "Vai Avanti": "Herencia europea (italiana)",
    "Vagabondo": "Herencia europea (italiana)",
    "Valentino": "Herencia europea (italiana)",
    "Valentino Bar": "Herencia europea (italiana)",
    "Valerio": "Herencia europea (italiana)",

    # ─────────────────────────────
    # French heritage
    # ─────────────────────────────
    "Atelier Fuerza": "Herencia europea (francesa)",
    "Bar La Cigale": "Herencia europea (francesa)",
    "Bar! Merci": "Herencia europea (francesa)",
    "Belier": "Herencia europea (francesa)",
    "Cafe Tabac": "Herencia europea (francesa)",
    "Café Maitre": "Herencia europea (francesa)",
    "Caprice": "Herencia europea (francesa)",
    "Chablis": "Herencia europea (francesa)",
    "Champagne Express": "Herencia europea (francesa)",
    "Champs Elysées": "Herencia europea (francesa)",
    "Chantilly": "Herencia europea (francesa)",
    "Co-Pain": "Herencia europea (francesa)",
    "Cocu": "Herencia europea (francesa)",
    "Colette": "Herencia europea (francesa)",
    "Côte Cafe": "Herencia europea (francesa)",
    "Gontran Cherrier": "Herencia europea (francesa)",
    "L'Abeille": "Herencia europea (francesa)",
    "L'Aperó": "Herencia europea (francesa)",
    "L'Artigiano": "Herencia europea (francesa)",
    "L´Aperó": "Herencia europea (francesa)",
    "La Francesa": "Herencia europea (francesa)",
    "La Sandwicherie": "Herencia europea (francesa)",
    "Lo del Francés": "Herencia europea (francesa)",
    "Méli Mélo": "Herencia europea (francesa)",
    "Nice": "Herencia europea (francesa)",
    "Niza": "Herencia europea (francesa)",
    "Paris": "Herencia europea (francesa)",
    "pret a porter": "Herencia europea (francesa)",
    "Provence": "Herencia europea (francesa)",
    "Riche": "Herencia europea (francesa)",
    "Rivière": "Herencia europea (francesa)",
    "Sabayon": "Herencia europea (francesa)",
    "Sablée": "Herencia europea (francesa)",
    "Sirop-Folie": "Herencia europea (francesa)",
    "Torre Paris": "Herencia europea (francesa)",
    "Trianon": "Herencia europea (francesa)",
    "Trois": "Herencia europea (francesa)",
    "Valence": "Herencia europea (francesa)",
    "Verne club": "Herencia europea (francesa)",
    "Victorica": "Herencia europea (francesa)",

    # ─────────────────────────────
    # Spanish heritage
    # ─────────────────────────────
    "Alcalá": "Herencia europea (española)",
    "Aribau": "Herencia europea (española)",
    "Asturias": "Herencia europea (española)",
    "Avila": "Herencia europea (española)",
    "Bar Iberia": "Herencia europea (española)",
    "Bar Soria": "Herencia europea (española)",
    "Benavente": "Herencia europea (española)",
    "Boiro": "Herencia europea (española)",
    "Chaval": "Herencia europea (española)",
    "Chupitos": "Herencia europea (española)",
    "Covadonga": "Herencia europea (española)",
    "El Roncal": "Herencia europea (española)",
    "Faro de Ons": "Herencia europea (española)",
    "Galerna": "Herencia europea (española)",
    "Gallego": "Herencia europea (española)",
    "Gijon": "Herencia europea (española)",
    "Gijón": "Herencia europea (española)",
    "Goya": "Herencia europea (española)",
    "Hispano": "Herencia europea (española)",
    "Iberia": "Herencia europea (española)",
    "Ibérico": "Herencia europea (española)",
    "La Barcelonesa": "Herencia europea (española)",
    "La Boqueria": "Herencia europea (española)",
    "La Giralda Cafetería": "Herencia europea (española)",
    "Lady Madrid": "Herencia europea (española)",
    "Madrid": "Herencia europea (española)",
    "Museo del jamón": "Herencia europea (española)",
    "Rincón Hispano": "Herencia europea (española)",
    "Rioja": "Herencia europea (española)",
    "Sagardi": "Herencia europea (española)",
    "Sánchez & Sánchez": "Herencia europea (española)",
    "Suárez": "Herencia europea (española)",
    "Taberna Baska": "Herencia europea (española)",
    "Tancat": "Herencia europea (española)",
    "Tapas De Lucía": "Herencia europea (española)",
    "Tegui": "Herencia europea (española)",
    "Toledo": "Herencia europea (española)",
    "Torija": "Herencia europea (española)",
    "Ultramarinos": "Herencia europea (española)",

    # ─────────────────────────────
    # Other European heritage
    # ─────────────────────────────
    "Auslander": "Herencia europea (otra)",
    "Avant Garten": "Herencia europea (otra)",
    "Baviera": "Herencia europea (otra)",
    "Belgica": "Herencia europea (otra)",
    "Berlin": "Herencia europea (otra)",
    "Berlina Bunker": "Herencia europea (otra)",
    "Bier Welt": "Herencia europea (otra)",
    "BierLife": "Herencia europea (otra)",
    "Bierhof": "Herencia europea (otra)",
    "Boa Lua": "Herencia europea (otra)",
    "Bodensee": "Herencia europea (otra)",
    "Boter": "Herencia europea (otra)",
    "Brač": "Herencia europea (otra)",
    "Breoghan": "Herencia europea (otra)",
    "Bucarest": "Herencia europea (otra)",
    "Bulgaria": "Herencia europea (otra)",
    "Cafezenda": "Herencia europea (otra)",
    "Celta Bar": "Herencia europea (otra)",
    "Checkpoint Charlie": "Herencia europea (otra)",
    "Confitera Saint Moritz": "Herencia europea (otra)",
    "Confitería Ritz": "Herencia europea (otra)",
    "El Holandés Cervecero": "Herencia europea (otra)",
    "Extrawurst": "Herencia europea (otra)",
    "Fritz": "Herencia europea (otra)",
    "Gibraltar": "Herencia europea (otra)",
    "Gott": "Herencia europea (otra)",
    "Gökotta": "Herencia europea (otra)",
    "Härlig": "Herencia europea (otra)",
    "Hereford": "Herencia europea (otra)",
    "Hönecker": "Herencia europea (otra)",
    "Iceland": "Herencia europea (otra)",
    "Kalimera Kafes": "Herencia europea (otra)",
    "Kefi": "Herencia europea (otra)",
    "Keller": "Herencia europea (otra)",
    "Keller Bier": "Herencia europea (otra)",
    "Keller Serrano": "Herencia europea (otra)",
    "Kellers": "Herencia europea (otra)",
    "La Greco": "Herencia europea (otra)",
    "La Helvetica": "Herencia europea (otra)",
    "Lagerhaus": "Herencia europea (otra)",
    "London City": "Herencia europea (otra)",
    "Londra": "Herencia europea (otra)",
    "Lutero Bar": "Herencia europea (otra)",
    "Nataria Portuguesa": "Herencia europea (otra)",
    "Nódica Smørrebrød": "Herencia europea (otra)",
    "Ogham": "Herencia europea (otra)",
    "Oslo": "Herencia europea (otra)",
    "Praga": "Herencia europea (otra)",
    "Quo Vadis": "Herencia europea (otra)",
    "Sláinte Irish Pub": "Herencia europea (otra)",
    "Suevia": "Herencia europea (otra)",
    "Taberna Odín": "Herencia europea (otra)",
    "Terminal Krakovia Bar": "Herencia europea (otra)",
    "Terranova": "Herencia europea (otra)",
    "Toller kaffeë": "Herencia europea (otra)",
    "Untertürkheim": "Herencia europea (otra)",
    "VALHALLA Bar Vikingo": "Herencia europea (otra)",
    "Valhalla Bar Vikingo": "Herencia europea (otra)",
    "Valk Taproom": "Herencia europea (otra)",
    "Varsovia": "Herencia europea (otra)",
    "Veikko": "Herencia europea (otra)",
    "Wunderbar": "Herencia europea (otra)",
    "York": "Herencia europea (otra)",
    "Öss": "Herencia europea (otra)",

    # ─────────────────────────────
    # Asian heritage
    # ─────────────────────────────
    "Baos": "Herencia asiática",
    "Bharat": "Herencia asiática",
    "Bi-Won": "Herencia asiática",
    "Budabar": "Herencia asiática",
    "Burma": "Herencia asiática",
    "Burman": "Herencia asiática",
    "Bushido Libros": "Herencia asiática",
    "Chinofino": "Herencia asiática",
    "Club M Omakase": "Herencia asiática",
    "Cochinchina": "Herencia asiática",
    "Dabbang": "Herencia asiática",
    "Dashi": "Herencia asiática",
    "Dashimaki": "Herencia asiática",
    "Delhi Darbar": "Herencia asiática",
    "Delhi Mahal": "Herencia asiática",
    "Delhi Masala": "Herencia asiática",
    "Dragon Doble": "Herencia asiática",
    "Fa Song Song": "Herencia asiática",
    "Fukuro": "Herencia asiática",
    "Fukuro Noodle bar": "Herencia asiática",
    "Ganesha - Arts & Pizza": "Herencia asiática",
    "Ginko": "Herencia asiática",
    "Haiku": "Herencia asiática",
    "Hakka": "Herencia asiática",
    "Haku Mikhuq": "Herencia asiática",
    "HANA Poke & Bar": "Herencia asiática",
    "Hiro": "Herencia asiática",
    "Irifune": "Herencia asiática",
    "Jeju": "Herencia asiática",
    "Jiro": "Herencia asiática",
    "Jisu": "Herencia asiática",
    "Kaido": "Herencia asiática",
    "Kaiteki": "Herencia asiática",
    "Katsu": "Herencia asiática",
    "Kiku": "Herencia asiática",
    "Kowloon": "Herencia asiática",
    "Krishna Veggie": "Herencia asiática",
    "Kumo": "Herencia asiática",
    "Kuro Kuma": "Herencia asiática",
    "Kyodo": "Herencia asiática",
    "La Causa Nikkei": "Herencia asiática",
    "Liangliang Barbecue": "Herencia asiática",
    "Lucky Dragon": "Herencia asiática",
    "Luxushi": "Herencia asiática",
    "Ma La Tang": "Herencia asiática",
    "Mahjong": "Herencia asiática",
    "Mandarín": "Herencia asiática",
    "MarSushi": "Herencia asiática",
    "Min Min": "Herencia asiática",
    "Mr. Ho": "Herencia asiática",
    "Mumbai Mahal": "Herencia asiática",
    "Namida": "Herencia asiática",
    "Okaeri": "Herencia asiática",
    "Pau Pei": "Herencia asiática",
    "Phuket": "Herencia asiática",
    "Q I Kai": "Herencia asiática",
    "Restaurant Wakayama": "Herencia asiática",
    "Restaurante Yugane": "Herencia asiática",
    "Shangai Express": "Herencia asiática",
    "SushiClub": "Herencia asiática",
    "Sushiclub": "Herencia asiática",
    "SushiLife": "Herencia asiática",
    "Suship": "Herencia asiática",
    "Taj Mahal": "Herencia asiática",
    "Takumi": "Herencia asiática",
    "Tandoor": "Herencia asiática",
    "Taré": "Herencia asiática",
    "Tayaki": "Herencia asiática",
    "Toki": "Herencia asiática",
    "Tori Tori": "Herencia asiática",
    "Umai Gourmet": "Herencia asiática",
    "Yuki": "Herencia asiática",
    "Yuzu": "Herencia asiática",

    # ─────────────────────────────
    # Middle Eastern heritage
    # ─────────────────────────────
    "Al Amal": "Herencia de Medio Oriente",
    "Alamut": "Herencia de Medio Oriente",
    "Armenia": "Herencia de Medio Oriente",
    "Barev": "Herencia de Medio Oriente",
    "Cafe Armenio": "Herencia de Medio Oriente",
    "El Nilo": "Herencia de Medio Oriente",
    "FAYER Buenos Aires": "Herencia de Medio Oriente",
    "Fayer": "Herencia de Medio Oriente",
    "Fenicia": "Herencia de Medio Oriente",
    "Garbis": "Herencia de Medio Oriente",
    "Küne": "Herencia de Medio Oriente",
    "Marroco SII": "Herencia de Medio Oriente",
    "Marrocos": "Herencia de Medio Oriente",
    "Shami": "Herencia de Medio Oriente",
    "Sobek Bowling & Drinks": "Herencia de Medio Oriente",
    "Soraya": "Herencia de Medio Oriente",

    # ─────────────────────────────
    # New Age
    # ─────────────────────────────
    "Chaman": "New Age / self help",
    "El Secreto de Oro": "New Age / self help",
    "Energía Libros": "New Age / self help",
    "Moksha": "New Age / self help",
    "Moksha café & vermú": "New Age / self help",
    "Praana": "New Age / self help",
    "Sueña": "New Age / self help",
    "Tauro": "New Age / self help",
    "Totem": "New Age / self help",
    "Tótem": "New Age / self help",
    "Triskel": "New Age / self help",

    # ─────────────────────────────
    # Animals and nature (neutral)
    # ─────────────────────────────
    "Las Palmas": "Animales y naturaleza (neutral)",
    "Las Palmeras": "Animales y naturaleza (neutral)",
    "Lazy Dog": "Animales y naturaleza (neutral)",
    "Lotos": "Animales y naturaleza (neutral)",
    "Osa Negra": "Animales y naturaleza (neutral)",
    "Owl": "Animales y naturaleza (neutral)",
    "Sauces": "Animales y naturaleza (neutral)",
    "Tilo": "Animales y naturaleza (neutral)",
    "Trigal": "Animales y naturaleza (neutral)",
    "Tropical": "Animales y naturaleza (neutral)",

    # ─────────────────────────────
    # Anglophone
    # ─────────────────────────────
    "Beer Conect": "Anglophone",
    "Beercoin": "Anglophone",
    "Famous Loungebar": "Anglophone",
    "Food Time": "Anglophone",
    "Fresh Kitchen Bar": "Anglophone",
    "Growlers": "Anglophone",
    "Growlers craft beer": "Anglophone",
    "Go Bar": "Anglophone",
    "Go in!": "Anglophone",
    "Gilmour Bar": "Anglophone",
    "Hawaii Love Bar": "Anglophone",
    "Hoboken": "Anglophone",
    "Hoyts": "Anglophone",
    "Jack the Ripper": "Anglophone",
    "J.W. Bradley": "Anglophone",
    "Kansas": "Anglophone",
    "Irish": "Anglophone",
    "La blonde": "Anglophone",
    "Lions Bar Cafe": "Anglophone",
    "NOLA": "Anglophone",
    "NYC": "Anglophone",
    "McCafé": "Anglophone",
    "Mr. Miga": "Anglophone",
    "Mr. Pedro": "Anglophone",
    "Mr. Sandwich": "Anglophone",
    "OK Food": "Anglophone",
    "Oh! Brothers": "Anglophone",
    "Ohana": "Anglophone",
    "Oh’No! Lulu": "Anglophone",
    "M Street Bar": "Anglophone",
    "On Tap": "Anglophone",
    "On Tap Craft Beer": "Anglophone",
    "Other Side": "Anglophone",
    "Plan B": "Anglophone",
    "Point": "Anglophone",
    "Post Street Bar": "Anglophone",
    "Proper restaurant": "Anglophone",
    "PublicBar": "Anglophone",
    "Rainbow": "Anglophone",
    "Red Monkey": "Anglophone",
    "Red Resto & Lounge": "Anglophone",
    "Remember": "Anglophone",
    "Restaurant King": "Anglophone",
    "Rising Sun": "Anglophone",
    "Rooftop Plaza de Mayo": "Anglophone",
    "Royal Gourmet": "Anglophone",
    "Royal Mansion": "Anglophone",
    "Stayhome": "Anglophone",
    "Social": "Anglophone",
    "Soho Club": "Anglophone",
    "Speed": "Anglophone",
    "Spring Brezee": "Anglophone",
    "Sweet": "Anglophone",
    "Sweety": "Anglophone",
    "Suit Suit": "Anglophone",
    "Taxi Bar": "Anglophone",
    "Tavern": "Anglophone",
    "Trade Sky Bar": "Anglophone",
    "Wine Bar Palermo": "Anglophone",
    "Winer": "Anglophone",
    "Winna": "Anglophone",
    "Woke": "Anglophone",
    "Whoopies": "Anglophone",
    "whoopies": "Anglophone",
    "Whoppies": "Anglophone",
    "Woopies": "Anglophone",
    "bule-bar": "Anglophone",
    "urban": "Anglophone",
    "victor Audio Bar": "Anglophone",
    "YOLO Bar": "Anglophone",
    "Bakery": "Anglophone",
    "Bonus track": "Anglophone",
    "Brooklyn Bakery": "Anglophone",
    "Down Town Matias": "Anglophone",
    "Downtown Matias": "Anglophone",
    "Downtown Deli": "Anglophone",
    "Garage Bar": "Anglophone",
    "Harmony Bar & Restaurante": "Anglophone",
    "Helka wine bar": "Anglophone",
    "Hunter Bier": "Anglophone",
    "Nice café/bakery": "Anglophone",
    "Ol'days": "Anglophone",
    "Olympo Sky Bar": "Anglophone",
    "Porto - Night Stadium": "Anglophone",
    "You Club": "Anglophone",
    "Zoom": "Anglophone",
    "Zoom Resto-Bar": "Anglophone",
    "i Fresh": "Anglophone",

    # ─────────────────────────────
    # Literary / intellectual / artistic
    # ─────────────────────────────
    "Adán Buenosayres Libros": "Literario / intelectual / artístico",
    "Agape Libros": "Literario / intelectual / artístico",
    "Café Libros del Pasaje": "Literario / intelectual / artístico",
    "Bebop Club": "Literario / intelectual / artístico",
    "Espacio INCAA KM 0 - Gaumont": "Literario / intelectual / artístico",
    "Estación Libro": "Literario / intelectual / artístico",
    "Eudeba": "Literario / intelectual / artístico",
    "Fetiche Libros": "Literario / intelectual / artístico",
    "Fondo de Cultura Económica": "Literario / intelectual / artístico",
    "Ifigenia Café Literario": "Literario / intelectual / artístico",
    "Fuerza Bruta": "Literario / intelectual / artístico",
    "Librería Biblos": "Literario / intelectual / artístico",
    "Librería Corneja": "Literario / intelectual / artístico",
    "Librería Helena de Buenos Aires": "Literario / intelectual / artístico",
    "Librería Lyris": "Literario / intelectual / artístico",
    "Librería del Alumno": "Literario / intelectual / artístico",
    "Librerías Levalle": "Literario / intelectual / artístico",
    "Librolandia": "Literario / intelectual / artístico",
    "Libros El Arca de Noé": "Literario / intelectual / artístico",
    "Mil Grullas Libros": "Literario / intelectual / artístico",
    "Monserrat lee libros": "Literario / intelectual / artístico",
    "Obel Libros": "Literario / intelectual / artístico",
    "Los 7 Locos": "Literario / intelectual / artístico",
    "Macondo": "Literario / intelectual / artístico",
    "Macondo Bar": "Literario / intelectual / artístico",
    "Mundo Alucinante": "Literario / intelectual / artístico",
    "Literal": "Literario / intelectual / artístico",
    "Paidos del Fondo": "Literario / intelectual / artístico",
    "Poe": "Literario / intelectual / artístico",
    "Poema 20": "Literario / intelectual / artístico",
    "Prólogo": "Literario / intelectual / artístico",
    "Recreo en Libros": "Literario / intelectual / artístico",
    "Textos": "Literario / intelectual / artístico",
    "Tierra de Libros": "Literario / intelectual / artístico",
    "Tomato Libros": "Literario / intelectual / artístico",
    "V&R Editoras S.A.": "Literario / intelectual / artístico",
    "Zama Libros": "Literario / intelectual / artístico",
    "Thelonious Club": "Literario / intelectual / artístico",
    "Verne club": "Literario / intelectual / artístico",
    "Tiempos Modernos": "Literario / intelectual / artístico",
    "Tecnica Artistica": "Literario / intelectual / artístico",
    "Sala Batato Barea": "Literario / intelectual / artístico",
    "Sala caras y caretas": "Literario / intelectual / artístico",
    "imaginario Cultural": "Literario / intelectual / artístico",
}


In [ ]:
# apply overrides
# --------------------------------------------------------------------

import pandas as pd

# load base enriched dataset (output from previous steps)
df = pd.read_csv("data/processed/osm_naming_ba_porteno_only_enriched.csv")

# create unified_label column by applying overrides
# if a name exists in manual_overrides, use that label
# otherwise, keep the original regex_label
df["unified_label"] = df["name"].map(manual_overrides).fillna(df["regex_label"])

# count how many overrides were applied
num_overrides = (df["name"].isin(manual_overrides)).sum()
print(f"applied {num_overrides} manual overrides")

# save as a CSV
output_path = "data/processed/osm_naming_ba_porteno_only_enriched_with_manual_overrides.csv"
df.to_csv(output_path, index=False)

print(f"\saved final dataset to: {output_path}")

# show summary of unified_label distribution
print("unified_label distribution")
label_counts = df["unified_label"].value_counts()
for label, count in label_counts.items():
    pct = (count / len(df)) * 100
    print(f"{label:<45} | {count:>5} ({pct:>5.1f}%)")

print(f"\ntotal categories: {df['unified_label'].nunique()}")
print(f"total venues: {len(df)}")


pplied 876 manual overrides
\saved final dataset to: data/processed/osm_naming_ba_porteno_only_enriched_with_manual_overrides.csv
unified_label distribution
Sin motivo claro / nombre genérico            |  2474 ( 50.3%)
Anglophone                                    |   441 (  9.0%)
Folklore y cultura local argentina            |   392 (  8.0%)
Herencia europea (italiana)                   |   297 (  6.0%)
Personal / familiar                           |   179 (  3.6%)
Herencia asiática                             |   154 (  3.1%)
Herencia europea (francesa)                   |   152 (  3.1%)
Animales y naturaleza (neutral)               |   138 (  2.8%)
Religioso / devocional                        |   133 (  2.7%)
Nostalgia rioplatense                         |   105 (  2.1%)
Culturas latinoamericanas vecinas             |    95 (  1.9%)
Literario / intelectual / artístico           |    88 (  1.8%)
Herencia europea (otra)                       |    86 (  1.7%)
Herencia europea (españ

## Step 7: Exporting Minimal Dataset for Web Visualization

**Goal:** Create a minimal venue dataset containing only essential fields needed for visualization.

**Technical Approach:**
- Load final enriched dataset (after manual review)
- Filter out venues with no motif (label "Sin motivo claro / nombre genérico")
- Select only essential columns:
  - OSM identifiers (type, id)
  - Geographic coordinates (lat, lon)
  - Administrative geography (barrio, comuna)
  - Cultural classification (unified_label)
  - Venue name (for popups)
- Save as compact CSV

**Output:** `places.csv` — minimal venue dataset for map

In [41]:
# create places.csv: minimal venue dataset for visualization
# --------------------------------------------------------------------

import pandas as pd

# load the final cleaned dataset
df_final = pd.read_csv("data/processed/osm_naming_ba_porteno_only_enriched_FINAL.csv")

# filter out rows with "Sin motivo claro / nombre genérico" as unified_label
df_final = df_final[df_final["unified_label"] != "Sin motivo claro / nombre genérico"]

print(f"loaded {len(df_final)} venues from final dataset")

# select essential columns for visualization
# only keep what's needed to display points on map with proper labels
essential_columns = [
    "osm_type",       
    "osm_id",         
    "key",            # OSM tag key (amenity/shop)
    "value",          # OSM tag value (cafe/bar/etc)
    "name",           # original venue name
    "lat",            # latitude
    "lon",            # longitude
    "barrio",         # neighborhood name
    "comuna",         # comuna number
    "_name_norm",     # normalized name (for matching/deduplication)
    "unified_label"   # final cultural motif classification
]

# create dataframe
df_places = df_final[essential_columns].copy()

# save as places.csv
output_path = "mapbox_geojson/places.csv"
df_places.to_csv(output_path, index=False)

print(f"\nsaved {len(df_places)} venues to {output_path}")
print(f"columns: {list(df_places.columns)}")

# display sample
print(df_places.head(1))


loaded 2555 venues from final dataset

saved 2555 venues to mapbox_geojson/places.csv
columns: ['osm_type', 'osm_id', 'key', 'value', 'name', 'lat', 'lon', 'barrio', 'comuna', '_name_norm', 'unified_label']
  osm_type      osm_id      key value             name        lat        lon  \
6     node  9576523317  amenity  cafe  Nuestra América -34.607275 -58.414512   

    barrio  comuna       _name_norm                      unified_label  
6  ALMAGRO     5.0  nuestra america  Culturas latinoamericanas vecinas  


## Step 8: Term Extraction with TF-IDF

**Goal:** Extract representative terms for each cultural motif using TF-IDF (Term Frequency-Inverse Document Frequency).

We compute TF-IDF independently for each motif to identify words that best characterize each cultural tradition. This per-motif approach avoids favoring large motifs over small ones, ensuring every motif gets its own distinctive vocabulary.
We analyze for venues within a motif:
- Term Frequency: How often does a word appear across venues in this motif?
- Inverse Document Frequency (IDF): Does the word appear in all venues (low IDF) or just some venues (high IDF)?

We average TF-IDF scores across venues to find consistently representative terms

**Technical Approach:**
1. For each motif, compute TF-IDF separately using only that motif's venues
2. Filter stopwords (Spanish, English, and some generic business terms)
3. Require words to appear in at least 3 venues (min_df=3)
4. Average TF-IDF scores across venues to find consistently representative terms
5. Extract top 10 words with highest average TF-IDF scores
6. Collapse adjacent tokens into 2–3 word phrases when a neighbor appears in ≥90% of that word's mentions (e.g. `croque madame`)
6. Save results as JSON with term counts and TF-IDF scores

**Output:** `motif_details.json` — motif metadata with distinctive terms


In [9]:
# step 8: term extraction with TF-IDF
# --------------------------------------------------------------------

import pandas as pd
import json
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

# load the final cleaned dataset
df_final = pd.read_csv("data/processed/osm_naming_ba_porteno_only_enriched_FINAL.csv")

# filter out chains (names appearing >8 times, assumed to be franchises)
df_final = df_final.groupby('name').filter(lambda x: len(x) <= 8)

# remove generic category
df_filtered = (
    df_final[df_final["unified_label"] != "Sin motivo claro / nombre genérico"]
    .reset_index(drop=True)
)

print(f"analyzing {len(df_filtered)} venues (non-generic, non-chain)")

# label translations: Spanish -> English
label_translations = {
    "Religioso / devocional": "Religious / Devotional",
    "Próceres nacionales y símbolos patrios": "National Heroes and Patriotic Symbols",
    "Culturas latinoamericanas vecinas": "Neighboring Latin American Cultures",
    "Herencia europea (italiana)": "Italian Heritage",
    "Herencia europea (francesa)": "French Heritage",
    "Herencia europea (española)": "Spanish Heritage",
    "Herencia europea (otra)": "Other European Heritage",
    "Herencia judía": "Jewish Heritage",
    "New Age / self help": "New Age",
    "Personal / familiar": "Personal / Familiar",
    "Nostalgia rioplatense": "Río de la Plata Nostalgia",
    "Herencia asiática": "Asian Heritage",
    "Herencia de Medio Oriente": "Middle Eastern Heritage",
    "Folklore y cultura local argentina": "Argentine Folklore and Local Culture",
    "Nombres comunitarios / unión / popular": "Community Names",
    "Animales y naturaleza (neutral)": "Animals and Nature",
    "Literario / intelectual / artístico": "Artistic / Intellectual",
    "Anglophone": "Anglophone",
    "Sin motivo claro / nombre genérico": "No Clear Motive / Generic Name",
}

# stopwords: combine English + Spanish + domain-specific terms
specific_stops = [
    # Spanish stopwords
    "y", "de", "la", "el", "que", "en", "los", "del", "se", "las",
    "un", "una", "por", "con", "para", "al", "es", "su", "lo",
    # Spanish business types
    "cafe", "cafeteria", "bar", "parrilla", "resto", "restobar",
    "restaurante", "cerveceria", "heladeria",
    "pizzeria", "confiteria", "vinoteca",
    "libreria", "panaderia", "pasteleria", "café",
    # English business types
    "coffee", "coffe", "burger", "pizza", "grill", "pub", "brew", "beer",
    "shop", "store", "food", "kitchen", "taste"
]

combined_stops = list(ENGLISH_STOP_WORDS) + specific_stops


import re
from collections import Counter

NGRAM_THRESHOLD = 0.90  # merge when neighbor appears in >= 90% of anchor mentions
MAX_NGRAM = 3


def tokenize_name(name):
    return re.findall(r"\b\w+\b", str(name).lower())


def find_phrase_spans(tokens, text_tokens):
    n = len(tokens)
    return [
        i
        for i in range(len(text_tokens) - n + 1)
        if text_tokens[i : i + n] == tokens
    ]


def phrase_contexts(texts, tokens):
    contexts = []
    for text in texts:
        toks = tokenize_name(text)
        for start in find_phrase_spans(tokens, toks):
            end = start + len(tokens) - 1
            contexts.append(
                {
                    "prev": toks[start - 1] if start > 0 else None,
                    "next": toks[end + 1] if end < len(toks) - 1 else None,
                }
            )
    return contexts


def dominant_adjacent(contexts, side):
    if not contexts:
        return None, 0.0
    key = "next" if side == "next" else "prev"
    neighbors = [c[key] for c in contexts if c[key] is not None]
    if not neighbors:
        return None, 0.0
    neighbor, count = Counter(neighbors).most_common(1)[0]
    return neighbor, count / len(contexts)


def build_phrase_from_unigram(word, texts, threshold=NGRAM_THRESHOLD, max_n=MAX_NGRAM):
    """Extend a top word into a 2-3 word phrase when a neighbor is >= threshold consistent."""
    tokens = [word]
    while len(tokens) < max_n:
        contexts = phrase_contexts(texts, tokens)
        if not contexts:
            break

        next_neighbor, next_ratio = dominant_adjacent(contexts, "next")
        prev_neighbor, prev_ratio = dominant_adjacent(contexts, "prev")

        if next_ratio >= threshold and next_neighbor:
            tokens.append(next_neighbor)
        elif prev_ratio >= threshold and prev_neighbor:
            tokens.insert(0, prev_neighbor)
        else:
            break

    phrase = " ".join(tokens)
    count = len(phrase_contexts(texts, tokens))
    return phrase, count, tokens


def collapse_top_terms(top_unigrams, texts, threshold=NGRAM_THRESHOLD, max_n=MAX_NGRAM, top_k=10):
    """Merge adjacent words into phrases; drop singles absorbed into a phrase."""
    collapsed = []
    used_words = set()

    for entry in top_unigrams:
        word = entry["word"]
        if word in used_words:
            continue

        phrase, count, token_list = build_phrase_from_unigram(
            word, texts, threshold=threshold, max_n=max_n
        )
        for tok in token_list:
            used_words.add(tok)

        collapsed.append(
            {
                "word": phrase,
                "tfidf": entry["tfidf"],
                "count": count,
            }
        )

    collapsed.sort(key=lambda x: x["tfidf"], reverse=True)
    return collapsed[:top_k]


# extract top terms per motif (per-motif TF-IDF)
# strategy: compute TF-IDF separately for each motif
# - allows small motifs to have their own distinctive vocabulary
# - fixed min_df=3: term must appear in at least 3 venues within motif

motif_details = {}
TOP_K = 10  # number of top terms to extract per motif
MIN_DF = 3  # fixed min_df for all motifs

for label in sorted(df_filtered["unified_label"].unique()):
    # get venue names for this motif only
    motif_venues = df_filtered[df_filtered["unified_label"] == label]
    texts = motif_venues["name"].tolist()
    total_venues = len(texts)
    try:
        # count vectorizer for this motif
        count_vec_uni = CountVectorizer(
            max_features=300,
            min_df=MIN_DF,
            max_df=0.80,
            ngram_range=(1, 1),
            stop_words=combined_stops
        )
        X_counts_uni = count_vec_uni.fit_transform(texts)
        
        # TF-IDF vectorizer for this motif
        tfidf_vec_uni = TfidfVectorizer(
            ngram_range=(1, 1),
            vocabulary=count_vec_uni.vocabulary_,
            use_idf=True,
            norm="l2"
        )
        X_tfidf_uni = tfidf_vec_uni.fit_transform(texts)
        
        # compute average TF-IDF and total counts
        # averaging finds words that consistently appear with high TF-IDF across venues
        tfidf_uni = X_tfidf_uni.mean(axis=0).A1
        counts_uni = X_counts_uni.sum(axis=0).A1
        features_uni = np.array(count_vec_uni.get_feature_names_out())
        
        # rank by TF-IDF
        top_uni_idx = np.argsort(tfidf_uni)[-(TOP_K * 2):][::-1]
        raw_unigrams = [
            {"word": features_uni[i], "tfidf": float(tfidf_uni[i]), "count": int(counts_uni[i])}
            for i in top_uni_idx
            if tfidf_uni[i] > 0
        ]
        top_unigrams = collapse_top_terms(raw_unigrams, texts, top_k=TOP_K)
    except ValueError:
        # handle case where vocabulary is empty (motif too small)
        top_unigrams = []
    
    # build motif entry
    motif_entry = {
        "label_es": label,
        "label_en": label_translations.get(label, label),
        "total_venues": total_venues,
        "top_unigrams": top_unigrams,
    }
    
    motif_details[label] = motif_entry

# save motif details as JSON
output_path = "mapbox_geojson/motif_details.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(motif_details, f, ensure_ascii=False, indent=2)

print(f"\nsaved motif details to: {output_path}")
print(f"number of motifs: {len(motif_details)}")

# print summary for each motif
for label, details in motif_details.items():
    print(f"\n{label} ({details['label_en']})")
    print(f"  total venues: {details['total_venues']}")
    print(f"  top {TOP_K} terms:")
    for u in details['top_unigrams']:
        print(f"    - {u['word']}: {u['count']} occurrences (TF-IDF: {u['tfidf']:.3f})")


analyzing 2384 venues (non-generic, non-chain)

saved motif details to: mapbox_geojson/motif_details.json
number of motifs: 18

Anglophone (Anglophone)
  total venues: 418
  top 10 terms:
    - bakery: 14 occurrences (TF-IDF: 0.028)
    - big: 9 occurrences (TF-IDF: 0.022)
    - house: 10 occurrences (TF-IDF: 0.021)
    - new: 6 occurrences (TF-IDF: 0.014)
    - dandy: 6 occurrences (TF-IDF: 0.014)
    - club: 6 occurrences (TF-IDF: 0.014)
    - green: 6 occurrences (TF-IDF: 0.012)
    - whoopies: 5 occurrences (TF-IDF: 0.012)
    - import: 5 occurrences (TF-IDF: 0.012)
    - craft: 8 occurrences (TF-IDF: 0.012)

Animales y naturaleza (neutral) (Animals and Nature)
  total venues: 171
  top 10 terms:
    - sol: 8 occurrences (TF-IDF: 0.047)
    - flores: 5 occurrences (TF-IDF: 0.029)
    - negra: 5 occurrences (TF-IDF: 0.025)
    - gallo: 4 occurrences (TF-IDF: 0.023)
    - campo: 4 occurrences (TF-IDF: 0.023)
    - cisne: 4 occurrences (TF-IDF: 0.023)
    - lobo: 4 occurrences (TF-IDF

## Step 9: Correlation Analysis — Motifs vs Socioeconomic Variables

**Goal:** Quantify relationships between naming patterns and comuna socioeconomic characteristics.

**Technical Approach:**
1. Build contingency table at comuna level (motif proportions per comuna)
2. Compute Pearson correlations between:
   - Motif proportions
   - Socioeconomic variables (income, average age)
3. Add correlation coefficients to motif metadata

**Output:** Updated `motif_details.json` with correlation coefficients

In [10]:
# step 9: compute correlations — motifs vs socioeconomic variables
# --------------------------------------------------------------------
#
# methodology: Pearson correlation coefficient (r) measures linear association
# between two continuous variables

import json
import pandas as pd
import numpy as np

# load final enriched dataset (already has socioeconomic data merged)
df = pd.read_csv(
    "data/processed/osm_naming_ba_porteno_only_enriched_FINAL.csv"
)

# remove generic category
df_filtered = df[df["unified_label"] != "Sin motivo claro / nombre genérico"].copy()

# filter out chains (names appearing >8 times, franchises)
df_filtered = df_filtered.groupby('name').filter(lambda x: len(x) <= 8)

# ensure comuna is numeric for merging
df_osm = df_filtered.copy()
df_osm["comuna"] = pd.to_numeric(df_osm["comuna"], errors="coerce")

print(f"computing correlations for {len(df_osm)} venues (non-generic, non-chain)")

# build contingency table: motif proportions per comuna
# count venues for each comuna-motif combination
ct_comuna_motivo = pd.crosstab(df_osm["comuna"], df_osm["unified_label"])

# calculate proportions: each row sums to 1
prop_comuna_motivo = ct_comuna_motivo.div(ct_comuna_motivo.sum(axis=1), axis=0)

# attach socioeconomic variables
# for each comuna, get socioeconomic values
socio_vars = ["ipcf_promedio_pesos", "edad_promedio_anios", "porc_65mas"]

for var in socio_vars:
    prop_comuna_motivo[var] = df_osm.groupby("comuna")[var].first()

print(f"attached socioeconomic variables to comuna-motif profile")

# compute correlations
# dictionary to hold correlation results
correlations = {}

# for each socioeconomic variable, compute correlation with each motif
for var in socio_vars:
    correlations[var] = prop_comuna_motivo.corrwith(prop_comuna_motivo[var])

print(f"computed correlations for {len(socio_vars)} socioeconomic variables")

# load and update motif details
# load existing motif_details.json
with open("mapbox_geojson/motif_details.json", "r", encoding="utf-8") as f:
    motif_details = json.load(f)

# add correlation data to each motif
for label in motif_details.keys():
    motif_details[label]["correlations"] = {
        var: round(correlations[var].get(label, 0), 3)
        for var in socio_vars
    }

# save updated motif_details.json
output_path = "mapbox_geojson/motif_details.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(motif_details, f, ensure_ascii=False, indent=2)

print(f"updated {output_path} with correlation data")

# print summary of correlations
for label, details in motif_details.items():
    print(f"\n{label} ({details['label_en']}):")
    for var in socio_vars:
        print(f"  {var}: {details['correlations'][var]}")


computing correlations for 2384 venues (non-generic, non-chain)
attached socioeconomic variables to comuna-motif profile
computed correlations for 3 socioeconomic variables
updated mapbox_geojson/motif_details.json with correlation data

Anglophone (Anglophone):
  ipcf_promedio_pesos: 0.721
  edad_promedio_anios: 0.686
  porc_65mas: 0.605

Animales y naturaleza (neutral) (Animals and Nature):
  ipcf_promedio_pesos: -0.102
  edad_promedio_anios: -0.035
  porc_65mas: 0.027

Culturas latinoamericanas vecinas (Neighboring Latin American Cultures):
  ipcf_promedio_pesos: -0.047
  edad_promedio_anios: -0.154
  porc_65mas: -0.103

Folklore y cultura local argentina (Argentine Folklore and Local Culture):
  ipcf_promedio_pesos: -0.572
  edad_promedio_anios: -0.707
  porc_65mas: -0.673

Herencia asiática (Asian Heritage):
  ipcf_promedio_pesos: 0.432
  edad_promedio_anios: 0.275
  porc_65mas: 0.287

Herencia de Medio Oriente (Middle Eastern Heritage):
  ipcf_promedio_pesos: 0.364
  edad_promedi

## Step 10: Finalizing Motif Metadata with Descriptions

**Objective:** Add human-readable descriptions to motif metadata for public display.

**Output:** Final `motif_details.json` with complete metadata (terms, correlations, descriptions)

In [11]:
# finalize motif details: add descriptions for visualization
# -------------------------------------------------------------

import json

# load motif_details.json with correlations (from previous cell)
with open("mapbox_geojson/motif_details.json", "r", encoding="utf-8") as f:
    motif_details = json.load(f)

# motif descriptions: explanations for each motif
motif_descriptions = {
    "Anglophone": 
        "English-language names that signal imported trends, global consumer culture, or aspirations toward internationalism.",
    
    "Animales y naturaleza (neutral)": 
        "References to animals, plants, or natural features used as neutral, descriptive branding.",
    
    "Culturas latinoamericanas vecinas": 
        "Allusions to neighboring Latin American countries: food, geography, colloquialisms, or regional identities within the Southern Cone.",
    
    "Folklore y cultura local argentina": 
        "Names rooted in Argentine and porteño identity, drawing on slang, fútbol, local customs, and recognizable cultural symbols.",
    
    "Herencia asiática": 
        "References to Asian cuisines or cultural motifs, spanning East, South, and Southeast Asia.",
    
    "Herencia de Medio Oriente": 
        "Names tied to Middle Eastern diasporic presence in Buenos Aires, reflected in food traditions, language, and family heritage.",
    
    "Herencia europea (española)": 
        "Spanish cultural references—from regional identities to gastronomy—that reflect long-standing historical ties.",

    "Herencia europea (francesa)": 
        "French cultural references including gastronomy and cultural motifs.",

    "Herencia europea (italiana)": 
        "Italian references connected to food, surnames, or cultural identity, reflecting the city’s significant Italian heritage.",
    
    "Herencia europea (otra)": 
        "Other European influences—German, Greek, Portuguese, Scandinavian—visible in food traditions, surnames, or cultural symbols.",

    "Herencia judía": 
        "Jewish cultural and religious heritage represented in community spaces and cultural identity.",

    "Literario / intelectual / artístico": 
        "Names invoking literature, art, philosophy, or intellectual identity.",
    
    "New Age / self help": 
        "Branding shaped by wellness, spirituality, mindfulness, or contemporary self-improvement language.",
    
    "Nombres comunitarios / unión / popular": 
        "Collective-oriented names referencing neighborhoods, solidarity, cooperatives, or working-class identity.",
    
    "Nostalgia rioplatense": 
        "Invocations of a Río de la Plata past: tango, cafés, and period aesthetics that evoke local nostalgia.",
    
    "Personal / familiar": 
        "Personal names and references to family or close relations.",
    
    "Próceres nacionales y símbolos patrios": 
        "References to national heroes and patriotic symbols embedded in Argentina’s history.",
    
    "Religioso / devocional": 
        "Religious references across Catholic, evangelical, and other traditions, foregrounding devotion, saints, or sacred spaces.",
}

# add descriptions to motif details
for label in motif_details.keys():
    motif_details[label]["description"] = motif_descriptions.get(label, "")

# save final motif_details.json
output_path = "mapbox_geojson/motif_details.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(motif_details, f, ensure_ascii=False, indent=2)

print(f"added descriptions to motif_details.json")


added descriptions to motif_details.json


## Step 11: Enriching Comuna Boundaries with Motif Data

**Goal:** Prepare comuna polygons for choropleth visualization by adding venue counts, motif distributions, and socioeconomic rankings.

**Technical Approach:**
- Load comuna boundary GeoJSON from the city's open data platform
- Compute motif counts per comuna from enriched venue data
- Calculate income rankings (1 = wealthiest comuna, 15 = lowest income)
- Add average age and other demographic indicators
- Attach all metrics as properties to each comuna feature
- Save enriched GeoJSON for Mapbox visualization

**Output:** `comunas_map.geojson` — comuna polygons with motif distributions and socioeconomic metadata

In [12]:
# step 11: enrich comunas.geojson — add motif counts and income rankings
# --------------------------------------------------------------------

import json
import pandas as pd

# df_filtered has:
# - non-generic motifs only
# - numeric comuna IDs
print(f"using {len(df_filtered)} venues (non-generic)")

# compute motif counts per comuna
# create contingency table: comunas (rows) x motifs (columns)
motif_counts_per_comuna = pd.crosstab(
    df_filtered["comuna"], 
    df_filtered["unified_label"]
)

# convert to dictionary format for GeoJSON properties
# format: {comuna_id: {motif: count, motif: count, ...}, ...}
motif_dict = motif_counts_per_comuna.to_dict(orient="index")

print(f"computed motif counts for {len(motif_dict)} comunas")
print(f"total motifs: {len(motif_counts_per_comuna.columns)}")

# compute income rankings and average age
# get one income value per comuna (all venues in a comuna have same value)
df_all = pd.read_csv(
    "data/processed/osm_naming_ba_porteno_only_enriched_FINAL.csv"
)

# income ranking
comuna_income = df_all.groupby("comuna")["ipcf_promedio_pesos"].first()
income_ranks = comuna_income.rank(method="min", ascending=False).astype(int)
income_rank_dict = income_ranks.to_dict()

print(f"computed income rankings for {len(income_rank_dict)} comunas")

# average age per comuna
comuna_age = df_all.groupby("comuna")["edad_promedio_anios"].first()
comuna_age = comuna_age.dropna()
age_dict = comuna_age.to_dict()

print(f"extracted average age for {len(age_dict)} comunas")

# load geojson file
# load the original comunas GeoJSON file
with open("data/comunas_data/comunas.geojson", "r", encoding="utf-8") as f:
    comunas_geojson = json.load(f)

print(f"loaded {len(comunas_geojson['features'])} comuna features from GeoJSON")

# add motif counts, income ranks, and average age to each comuna feature
for feature in comunas_geojson["features"]:
    props = feature["properties"]
    comuna_id = int(props.get("comuna"))
    
    # add motif counts and total venues
    props["motif_counts"] = motif_dict[comuna_id]
    props["total_venues"] = sum(motif_dict[comuna_id].values())
    
    # add income rank
    props["income_rank"] = income_rank_dict[comuna_id]
    
    # add average age
    props["edad_promedio"] = age_dict[comuna_id]

print(f"enriched {len(comunas_geojson['features'])} comuna features with demographics")

# save enriched geojson
# save to mapbox_geojson folder for web viz
output_geojson_path = "mapbox_geojson/comunas_map.geojson"

with open(output_geojson_path, "w", encoding="utf-8") as f:
    json.dump(comunas_geojson, f, ensure_ascii=False, indent=2)

print(f"\nsaved enriched GeoJSON to {output_geojson_path}")

# print sample output
print("sample: motif counts, income rank, and average age for first 3 comunas")

for i, feature in enumerate(comunas_geojson["features"][:3]):
    props = feature["properties"]
    print(f"\nComuna {props.get('comuna')}:")
    print(f"  income rank: {props.get('income_rank', 'N/A')} (1 = highest)")
    print(f"  average age: {props.get('edad_promedio', 'N/A')} years")
    print(f"  total venues: {props.get('total_venues', 0)}")
    motif_counts = props.get("motif_counts", {})
    
    # show top 5 motifs
    sorted_motifs = sorted(
        motif_counts.items(), 
        key=lambda x: x[1], 
        reverse=True
    )[:5]
    
    for motif, count in sorted_motifs:
        pct = (count / props.get("total_venues", 1)) * 100
        print(f"    {motif}: {count} venues ({pct:.1f}%)")

print("comuna.geojson enrichment complete")

using 2384 venues (non-generic)
computed motif counts for 15 comunas
total motifs: 18
computed income rankings for 15 comunas
extracted average age for 15 comunas
loaded 15 comuna features from GeoJSON
enriched 15 comuna features with demographics

saved enriched GeoJSON to mapbox_geojson/comunas_map.geojson
sample: motif counts, income rank, and average age for first 3 comunas

Comuna 1:
  income rank: 9 (1 = highest)
  average age: 37.8 years
  total venues: 533
    Anglophone: 103 venues (19.3%)
    Herencia europea (italiana): 58 venues (10.9%)
    Folklore y cultura local argentina: 50 venues (9.4%)
    Literario / intelectual / artístico: 49 venues (9.2%)
    Herencia asiática: 35 venues (6.6%)

Comuna 2:
  income rank: 1 (1 = highest)
  average age: 42.5 years
  total venues: 209
    Anglophone: 35 venues (16.7%)
    Herencia europea (italiana): 32 venues (15.3%)
    Herencia europea (francesa): 21 venues (10.0%)
    Folklore y cultura local argentina: 20 venues (9.6%)
    Anima